<div dir="rtl">

# DDPM شرطیِ AZ80 — نوتبوک اصلاح‌شده و نقشه‌دار

**مقاله:** E. Azqadan, H. Jahed, A. Arami، «Predictive microstructure image generation using denoising diffusion probabilistic models»، *Acta Materialia* 261 (2023) 119406.

این نسخهٔ اصلاح‌شدهٔ `az80_image_generation_colab.ipynb` است و دو هدف دارد:

1. هر بخش کد را به بخش، شکل یا معادلهٔ مشخصی از مقاله وصل می‌کند تا بتوانی کد و مقاله را کنار هم بخوانی.
2. باگ‌ها و ناسازگاری‌های کد اصلی با مقاله را اصلاح می‌کند.

### برچسب‌های داخل کد
بالای هر سلول کد یک بلوک `PAPER MAP` هست و کنار خطوط مهم برچسب‌های زیر آمده است:

| برچسب | معنی |
|---|---|
| `[PAPER §x]`، `[PAPER Eq.n]`، `[PAPER Fig.n]` | مستقیماً از متن، معادله یا شکل مقاله |
| `[CODE-ONLY]` | در مقاله نیامده؛ از کد نویسندگان یا جزئیات پیاده‌سازی است |
| `[ASSUMPTION]` | مقاله چیزی نگفته و این انتخاب من است |
| `[ENGINEERING]` | فقط برای کم‌کردن حافظه یا زمان است و به مقاله ربطی ندارد |
| `[FIX]` | اصلاح نسبت به نوتبوک اصلی |

### چه چیزی را اصلاح کرده‌ام

| مورد | نوتبوک اصلی | این نسخه | دلیل |
|---|---|---|---|
| حلقهٔ داده | `itertools.cycle(train_loader)` دسته‌های دور اول را ذخیره و بعد فقط همان‌ها را تکرار می‌کرد؛ برش، قرینه‌سازی و shuffle فقط یک بار اعمال می‌شد | DataLoader در هر epoch از نو اجرا می‌شود | مقاله برش و قرینه‌سازی را در هر تکرار می‌گوید (§2.2.1) |
| معنای «iteration» | ۳۶۰۰ گام بهینه‌ساز، یعنی حدود ۳۸ epoch | پیش‌فرض: ۳۶۰۰ epoch (`ITERATION_UNIT='epoch'`)؛ با `'step'` به حالت قبل برمی‌گردد | ۱۳۰ ساعت تقسیم بر ۳۶۰۰ گام می‌شود ۱۳۰ ثانیه برای هر گام، که غیرممکن است. این **استنتاج من** است، نه تصریح مقاله |
| خودتوجهی | بلوک‌های ۱ و ۲ کدگذار بدون توجه، بلوک ۳ با دو سر، توجه در چهار بلوک کدگشا | چهار سر بعد از هر شش نمونه‌کاهی کدگذار؛ کدگشا بدون توجه؛ هر دو قابل تنظیم‌اند | §2.2.3: «four heads … after each down-sampling layer» |
| ذخیرهٔ checkpoint | checkpoint قبلی را کم‌حجم می‌کرد و بعد نسخهٔ جدید را می‌نوشت؛ اگر نوشتن شکست می‌خورد هیچ checkpoint کاملی نمی‌ماند | اول نسخهٔ جدید نوشته می‌شود؛ خطای نوشتن آموزش را نمی‌کشد؛ resume فقط از checkpoint کامل | جلوگیری از از دست رفتن آموزش چندروزه |
| cache تصاویر شکل‌ها | کلید cache مدل را در نظر نمی‌گرفت و بعد از آموزش مجدد تصاویر مدل قبلی را برمی‌گرداند | اثرانگشت وزن‌ها در کلید cache است | جلوگیری از شکل‌های قدیمی |
| `cfg_scale` | مسیر `cfg_scale > 0` خطا می‌داد | حذف شد | مقاله از classifier-free guidance استفاده نمی‌کند |
| batch size | ۴ | ۲ | کد اصلی `ddpm.py` ۲ بود و مقاله چیزی نمی‌گوید (`[ASSUMPTION]`) |
| پوشهٔ خروجی | همه در یک پوشه | اجرای `SMOKE_TEST` پوشهٔ جدا دارد | resume خودکار نباید از checkpoint آزمایشی ادامه بدهد |
| ساختار مدل | ۷ جفت لایه برای هر شرط در هر بلوک `Down` و `Up` تکرار شده بود | یک ماژول `ConditionTensors` با همان ریاضی | خوانایی |

**هشدار سازگاری:** چون ساختار توجه و نام لایه‌ها عوض شده، checkpoint آموزش‌دیدهٔ نویسندگان را نمی‌شود مستقیم در این مدل بارگذاری کرد.

### چه چیزی آزموده شده و چه چیزی نه
**آزموده شد (روی CPU، با دادهٔ ساختگی در همین قالب پوشه‌ها و با `SMOKE_TEST = True`):** خواندن داده و برچسب‌ها، آموزش در حالت epoch و step، ذخیرهٔ checkpoint و hard link، resume خودکار، محافظ ناسازگاری یکای تکرار، رفتار هنگام پر شدن دیسک، نمونه‌برداری، منحنی FID و هر سه شکل بخش ۱۰. منطق ذخیره و کم‌حجم‌سازی checkpoint جداگانه با یک مدل کوچک آزمون شد. مدل ۵۱۲×۵۱۲ نیز ساخته و تعداد پارامترهایش شمرده شد (حدود ۳۷۶ میلیون که ۸۲٪ آن هدهای شرط است).

**آزموده نشد:** اجرا روی GPU و مسیر fp16 (`AMP`)، دادهٔ واقعی، اتصال Drive، حافظهٔ GPU در اندازهٔ ۵۱۲، و آموزش طولانی. نتیجهٔ آزمون ساختگی معنی علمی ندارد؛ فقط نشان می‌دهد مسیر کامل کد بدون خطا اجرا می‌شود.

### نحوهٔ اجرا (Colab)
1. پوشهٔ داده را با همان ساختار نوتبوک اصلی (`Training/`، `Testing/`، `Training Labels.xlsx`، `Testing Labels.xlsx`) در My Drive بگذار.
2. در Colab برای GPU تنظیم کن: Runtime ← Change runtime type.
3. سلول‌ها را به ترتیب اجرا کن. اگر می‌خواهی اول فقط ببینی همه‌چیز بدون خطا اجرا می‌شود، در سلول تنظیمات `SMOKE_TEST = True` بگذار (تصاویر ۶۴×۶۴، ۵۰ گام انتشار، ۲ epoch).
4. آموزش کامل (۳۶۰۰ epoch) در یک جلسهٔ Colab جا نمی‌شود. checkpointها روی Drive ذخیره می‌شوند و `RESUME_FROM = 'auto'` در جلسهٔ بعد ادامه می‌دهد.

### آنچه این نوتبوک پیاده نمی‌کند
اندازه‌گیری درصد فاز Mg₁₇Al₁₂، درصد نواحی DRX و اندازهٔ دانه (شکل ۵ و §3.2 مقاله) در این نوتبوک نیست؛ مقاله این اندازه‌گیری‌ها را روی تصاویر انجام داده است.

</div>

<div dir="rtl">

## نقشهٔ کلی: کدام بخش نوتبوک ↔ کدام بخش مقاله

| بخش نوتبوک | چه می‌کند | بخش مقاله |
|---|---|---|
| ۲. تنظیمات | همهٔ هایپرپارامترها | §2.2.1، §2.2.3، §3.1 |
| ۳. `COND_VOCAB`، `ConditionTensors` | ۷ پارامتر فرایند ← جاسازی ← نگاشت خطی ← ۷ کانال | §2.1، §2.2.1، §2.2.3 (پاراگراف دوم) |
| ۳. `DoubleConv`، `Down`، `Up` | چهار کانولوشن + GroupNorm + GELU در هر بلوک، ادغام بیشینه، افزایش مقیاس دوخطی | §2.2.3 |
| ۳. `SelfAttention` | خودتوجهی چهارسر | §2.2.3 |
| ۳. `pos_encoding` | جاسازی موقعیتی گام زمانی | §2.2.3 |
| ۳. `EMA` | میانگین متحرک نمایی وزن‌ها | §2.2.3 |
| ۴. `Diffusion.__init__` | β، α و ᾱ | معادلهٔ ۱ و ۲ |
| ۴. `noise_images` | تصویر نویزی در گام دلخواه | معادلهٔ ۳ |
| ۴. `sample` | حذف تدریجی نویز | معادلهٔ ۴ و ۵ |
| ۵. داده | برش ۵۱۲، نرمال‌سازی به [−۱، ۱]، قرینه‌سازی، تفکیک ۱۱۴ به ۱۷ کلاس | §2.2.1، §2.2.4 |
| ۶ و ۷. آموزش | Adam، تابع زیان MSE | معادلهٔ ۶، §2.2.3 |
| ۸. نمونه‌برداری | تولید تصویر برای شرایط دیده‌شده و دیده‌نشده | §3.1 |
| ۱۰-۱ | منحنی FID و تحول یک تصویر در آموزش | شکل ۲، §2.2.4 |
| ۱۰-۲ | تصویر مصنوعی و واقعی، شرایط دیده‌شده | شکل ۳ |
| ۱۰-۳ | تصویر مصنوعی و واقعی، شرایط دیده‌نشده | شکل ۴ |

</div>

<div dir="rtl">

## ۱. راه‌اندازی

کتابخانه‌ها، انتخاب دستگاه (GPU یا CPU)، اتصال Google Drive در Colab و تثبیت seed. این بخش به بخش خاصی از مقاله مربوط نیست.

</div>

In [ ]:
# ======================================================================
# PAPER MAP | environment setup (no paper section)
# [ENGINEERING] imports, Google Drive mount (Colab), device, random seed
# ======================================================================

import copy
import itertools
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

try:
    import openpyxl  # noqa: F401
except ImportError:
    import sys
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openpyxl'], check=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device} | IN_COLAB={IN_COLAB}')

SEED = 42


def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()

<div dir="rtl">

## ۲. تنظیمات یکجا

همهٔ هایپرپارامترها اینجا جمع شده‌اند و کنار هرکدام نوشته‌ام از کجا آمده است. مقاله فقط بعضی از آن‌ها را صریح گفته: اندازهٔ برش ۵۱۲، چهار سر توجه، Adam و MSE و تعداد ۳۶۰۰ «iteration». بقیه (مثل T برابر ۱۰۰۰، نرخ یادگیری و batch size) از کد نویسندگان یا فرض من هستند.

**طول آموزش (`ITERATION_UNIT`).** مقاله می‌گوید پس از ۳۶۰۰ iteration و بیش از ۱۳۰ ساعت آموزش متوقف شد.

- اگر iteration یک گام بهینه‌ساز باشد، هر گام حدود ۱۳۰ ثانیه می‌شود. برای این مدل روی GPU از نوع P100 این عدد غیرممکن است.
- اگر iteration یک epoch باشد (یک دور کامل روی حدود ۳۷۷ تصویر آموزشی؛ با batch برابر ۲ حدود ۱۸۹ گام)، کل آموزش حدود ۶۸۰ هزار گام و هر گام حدود ۰٫۷ ثانیه می‌شود که معقول است.

بنابراین پیش‌فرض را `'epoch'` گذاشتم. این **استنتاج من** است و مقاله صریح نگفته. با `ITERATION_UNIT = 'step'` می‌توانی به معنای نوتبوک اصلی برگردی.

**خودتوجهی.** مقاله فقط می‌گوید «پس از هر لایهٔ نمونه‌کاهی» توجه چهارسر دارد و برای کدگشا چیزی نمی‌گوید؛ به همین دلیل `ATTN_DECODER_BLOCKS` خالی است. نویسندگان در کد خود توجه را در بلوک‌های کدگشای ۶ تا ۳ هم به کار برده‌اند.

</div>

In [ ]:
# ======================================================================
# PAPER MAP | Sections 2.2.1 / 2.2.3 / 3.1 -- all hyper-parameters in one place
# Tags: [PAPER] stated in the paper | [CODE-ONLY] only in the authors' code
#       [ASSUMPTION] my choice | [ENGINEERING] memory/speed only
# ======================================================================


SMOKE_TEST = False   # True -> tiny dry run (64x64 crops, 50 diffusion steps, 2 epochs) to check that everything runs

# ---- Images ------------------------------------------------------------------
IMAGE_SIZE = 64 if SMOKE_TEST else 512        # [PAPER 2.2.1] random 512x512 crops of the 1024x768 SEM images
assert IMAGE_SIZE % 64 == 0, 'the U-Net halves the image 6 times'

# ---- Diffusion (Eq. 1-6) -----------------------------------------------------
NOISE_STEPS = 50 if SMOKE_TEST else 1000      # [CODE-ONLY] T = 1000 (the paper gives no value)
BETA_START = 1e-4                             # [CODE-ONLY] linear variance schedule: beta_1 small ...
BETA_END = 0.02                               # [CODE-ONLY] ... up to beta_T (paper: "beta_1 is small, beta_T is large")

# ---- Network (2.2.3) ---------------------------------------------------------
ATTN_HEADS = 4                                # [PAPER 2.2.3] self-attention "with four heads"
ATTN_ENCODER_BLOCKS = (1, 2, 3, 4, 5, 6)      # [PAPER 2.2.3] "applied after each down-sampling layer"
ATTN_DECODER_BLOCKS = ()                      # [PAPER] no decoder attention is described.
                                              # Authors' code used (6, 5, 4, 3) -- see the markdown above.
                                              # Decoder blocks are named as in the authors' code: 6 = first/deepest.
EMBEDDING_DIM = 100                           # [CODE-ONLY] length of each process-parameter embedding vector

# ---- Optimisation (2.2.3) ----------------------------------------------------
LEARNING_RATE = 3e-4                          # [CODE-ONLY] paper: "Adam optimizer", no value given
BATCH_SIZE = 2                                # [ASSUMPTION] original ddpm.py used 2; the paper does not say
EMA_BETA = 0.995                              # [CODE-ONLY] paper only says EMA is used
EMA_STEP_START = 5 if SMOKE_TEST else 2000    # [CODE-ONLY] EMA starts after this many OPTIMISER steps
AMP_REQUESTED = True                          # [ENGINEERING] mixed precision (fp16) on GPU, saves memory

# ---- Training length (3.1: "after 3600 iterations ... more than 130 h") -------
ITERATION_UNIT = 'epoch'                      # 'epoch' (my reading of the paper) or 'step' (original notebook)
NUM_ITERATIONS = 2 if SMOKE_TEST else 3600    # [PAPER 3.1]
CHECKPOINT_EVERY = 1 if SMOKE_TEST else 600   # [PAPER Fig.2] panels c..h = 600, 1200, ..., 3600
EXTRA_CHECKPOINT_ITERS = () if SMOKE_TEST else (30,)   # [PAPER Fig.2] panel b = iteration 30
SAMPLE_EVERY = 2 if SMOKE_TEST else 300       # [ENGINEERING] progress images (each costs ~T U-Net passes)
LOG_EVERY = 1 if SMOKE_TEST else 50           # [ENGINEERING]
SAFE_CHECKPOINT_ORDER = True                  # [FIX] write the new checkpoint BEFORE slimming the old ones

# ---- The 7 process parameters (2.1, 2.2.1) -----------------------------------
# The order is the order of the label columns and of the model's 7 inputs.
CONDITION_COLUMNS = ['shape', 'location', 'cooling_rate', 'soaking',
                     'heat_treatment', 'forging_temp', 'magnification']
COND_VOCAB = {            # number of distinct codes per parameter -> size of each nn.Embedding table
    'shape': 2,           # 0 cylinder, 1 pre-form
    'location': 3,        # 0 tall flange, 1 web, 2 short flange
    'cooling_rate': 3,    # 0 1.5, 1 6, 2 10.4 C/s
    'soaking': 3,         # 0 normal (3 h), 1 1.5 h@350C + 1.5 h@250C, 2 2 h@350C + 1 h@250C
    'heat_treatment': 2,  # 0 none, 1 homogenised (420 C, 24 h)
    'forging_temp': 3,    # 0 250, 1 300, 2 350 C
    'magnification': 6,   # 0 100x, 1 500x, 2 1000x, 3 1500x, 4 2000x, 5 3000x
}
print(f'SMOKE_TEST={SMOKE_TEST} | image {IMAGE_SIZE}px | T={NOISE_STEPS} | batch {BATCH_SIZE} | '
      f'{NUM_ITERATIONS} {ITERATION_UNIT}s')

<div dir="rtl">

## ۳. معماری مدل (§2.2.3)

مقاله برای برآورد نویز $\epsilon_t$ از یک U-Net استفاده می‌کند. هر بلوک کدگذار شامل ادغام بیشینه، چهار لایهٔ کانولوشن، نرمال‌سازی گروهی و تابع GELU است و کدگشا ساختاری آینه‌ای با افزایش مقیاس دوخطی و اتصال‌های جهشی دارد. مدل بر دو چیز شرطی می‌شود:

- **گام زمانی** با جاسازی موقعیتی ترنسفورمر که به هر بلوک **افزوده** می‌شود (جمع).
- **هفت پارامتر فرایند** با روش خود مقاله: در هر بلوک، هر پارامتر یک جاسازی (`nn.Embedding`) می‌گیرد، با یک لایهٔ خطی به یک نقشهٔ دوبعدی هم‌اندازهٔ همان بلوک تبدیل می‌شود و به‌عنوان کانال اضافه **الحاق** می‌شود. نویسندگان روش رایج «جمع جاسازی کلاس با جاسازی زمانی» را امتحان کردند و ناکارآمد یافتند.

در کد هر شرط یک کانال می‌شود؛ پس هفت کانال آخر خروجی هر بلوک نقشهٔ شرط‌هاست (مثلاً `Down1`: ۲۵ کانال ویژگی + ۷ کانال شرط = ۳۲).

| بلوک | وضوح خروجی (برای ۵۱۲) | کانال خروجی | خودتوجهی (پیش‌فرض) |
|---|---|---|---|
| `inc` | ۵۱۲ | ۱۶ | — |
| کدگذار ۱ تا ۶ | ۲۵۶، ۱۲۸، ۶۴، ۳۲، ۱۶، ۸ | ۳۲، ۶۴، ۱۲۸، ۲۵۶، ۵۱۲، ۵۱۲ | چهار سر در هر شش بلوک |
| گلوگاه | ۸ | ۵۱۲ | — |
| کدگشا ۶ تا ۱ | ۱۶، ۳۲، ۶۴، ۱۲۸، ۲۵۶، ۵۱۲ | ۲۵۶، ۱۲۸، ۶۴، ۳۲، ۱۶، ۸ | ندارد |

`inc` (یک `DoubleConv` اولیه) و سه `DoubleConv` گلوگاه در متن مقاله نیامده‌اند و جزئیات پیاده‌سازی هستند. اتصال جهشی بلوک $k$ کدگشا خروجی بلوک $k-1$ کدگذار است (برای $k=1$ خروجی `inc`).

</div>

In [ ]:
# ======================================================================
# PAPER MAP | Section 2.2.3 'Model implementation' -- U-Net noise predictor eps_theta(x_t, t)
# Paper: 6 encoder blocks (max-pool + 4 conv + GroupNorm + GELU) with 4-head self-attention,
#        6 decoder blocks (bilinear up-sampling + 4 conv + GroupNorm + GELU) with skip connections,
#        time step ADDED to every block, the 7 process parameters CONCATENATED in every block.
# ======================================================================


COND_SIZES = [COND_VOCAB[c] for c in CONDITION_COLUMNS]
NUM_COND = len(COND_SIZES)   # 7 extra channels per block

# [ENGINEERING] Always use PyTorch's standard attention path (F.multi_head_attention_forward ->
# scaled_dot_product_attention, memory O(tokens)). In eval mode PyTorch may otherwise pick a "fast path" that,
# on CPU, builds the full tokens x tokens matrix (65,536^2 floats per head for the first encoder block).
try:
    torch.backends.mha.set_fastpath_enabled(False)
except AttributeError:   # very old PyTorch: nothing to switch off
    pass


class EMA:
    """[PAPER 2.2.3] exponential moving average of the weights, used to stabilise training.
    [CODE-ONLY] for the first `step_start_ema` optimiser steps the EMA copy simply follows the model."""

    def __init__(self, beta):
        super().__init__()
        self.beta = beta
        self.step = 0

    def update_model_average(self, ma_model, current_model):
        for current_params, ma_params in zip(current_model.parameters(), ma_model.parameters()):
            old_weight, up_weight = ma_params.data, current_params.data
            ma_params.data = self.update_average(old_weight, up_weight)

    def update_average(self, old, new):
        if old is None:
            return new
        return old * self.beta + (1 - self.beta) * new

    def step_ema(self, ema_model, model, step_start_ema=2000):
        if self.step < step_start_ema:
            self.reset_parameters(ema_model, model)
            self.step += 1
            return
        self.update_model_average(ema_model, model)
        self.step += 1

    def reset_parameters(self, ema_model, model):
        ema_model.load_state_dict(model.state_dict())


class SelfAttention(nn.Module):
    """[PAPER 2.2.3] multi-head self-attention to model long-range dependencies between image regions.
    Every pixel of the feature map is one token, so a (B, C, size, size) map becomes (B, size*size, C).
    [CODE-ONLY] pre-LayerNorm, residual connections and a small feed-forward block."""

    def __init__(self, channels, size, heads=4):
        super().__init__()
        self.channels = channels
        self.size = size
        self.mha = nn.MultiheadAttention(channels, heads, batch_first=True)
        self.ln = nn.LayerNorm([channels])
        self.ff_self = nn.Sequential(
            nn.LayerNorm([channels]),
            nn.Linear(channels, channels),
            nn.GELU(),
            nn.Linear(channels, channels),
        )

    def forward(self, x):
        x = x.view(-1, self.channels, self.size * self.size).swapaxes(1, 2)
        x_ln = self.ln(x)
        # [ENGINEERING] need_weights=False: the (tokens x tokens) attention matrix is never used, and
        # skipping it lets PyTorch use its memory-efficient kernel. The output is mathematically identical.
        attention_value, _ = self.mha(x_ln, x_ln, x_ln, need_weights=False)
        attention_value = attention_value + x
        attention_value = self.ff_self(attention_value) + attention_value
        return attention_value.swapaxes(2, 1).view(-1, self.channels, self.size, self.size)


class DoubleConv(nn.Module):
    """[PAPER 2.2.3] two 3x3 convolutions with Group Normalisation and GELU.
    Two DoubleConvs per block give the paper's 'four convolutional layers'."""

    def __init__(self, in_channels, out_channels, mid_channels=None, residual=False):
        super().__init__()
        self.residual = residual
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(1, mid_channels),
            nn.GELU(),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(1, out_channels),
        )

    def forward(self, x):
        if self.residual:
            return F.gelu(x + self.double_conv(x))
        return self.double_conv(x)


class ConditionTensors(nn.Module):
    """[PAPER 2.2.3, paragraph 2] 'the embedded vectors corresponding to process parameters are linearly
    mapped into separate tensors with the same size of the corresponding layer spatial dimension and are
    then concatenated with feature maps of that layer'.

    For each of the 7 parameters:  integer code -> nn.Embedding -> nn.Linear -> (B, 1, H, W) map.
    The 7 maps are concatenated to the feature map as 7 extra channels."""

    def __init__(self, imsize):
        super().__init__()
        self.imsize = imsize
        self.heads = nn.ModuleList([
            nn.Sequential(nn.Embedding(n, EMBEDDING_DIM), nn.Linear(EMBEDDING_DIM, imsize * imsize))
            for n in COND_SIZES])

    def forward(self, x, labels):
        h, w = x.shape[-2:]
        assert h == w == self.imsize, f'feature map is {h}x{w}, expected {self.imsize}x{self.imsize}'
        maps = [head(lab).view(x.shape[0], 1, h, w) for head, lab in zip(self.heads, labels)]
        return torch.cat([x] + maps, dim=1)


class Down(nn.Module):
    """Encoder block: max-pool -> 4 conv layers -> concatenate the 7 condition maps -> add time embedding."""

    def __init__(self, in_channels, out_channels, imsize, emb_dim=512):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, in_channels, residual=True),
            DoubleConv(in_channels, out_channels - NUM_COND),   # the last 7 channels are the condition maps
        )
        self.cond = ConditionTensors(imsize)
        self.emb_layer = nn.Sequential(nn.SiLU(), nn.Linear(emb_dim, out_channels))

    def forward(self, x, t, labels):
        x = self.maxpool_conv(x)
        x = self.cond(x, labels)                       # [PAPER 2.2.3] concatenation of process parameters
        return x + self.emb_layer(t)[:, :, None, None]  # [PAPER 2.2.3] time step is ADDED (broadcast over H, W)


class Up(nn.Module):
    """Decoder block: bilinear up-sampling -> concatenate skip connection -> 4 conv layers ->
    concatenate the 7 condition maps -> add time embedding."""

    def __init__(self, in_channels, out_channels, imsize, emb_dim=512):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = nn.Sequential(
            DoubleConv(in_channels, in_channels, residual=True),
            DoubleConv(in_channels, out_channels - NUM_COND, in_channels // 2),
        )
        self.cond = ConditionTensors(imsize)
        self.emb_layer = nn.Sequential(nn.SiLU(), nn.Linear(emb_dim, out_channels))

    def forward(self, x, skip_x, t, labels):
        x = self.up(x)
        x = torch.cat([skip_x, x], dim=1)              # [PAPER 2.2.3] skip connection = concatenation
        x = self.conv(x)
        x = self.cond(x, labels)
        return x + self.emb_layer(t)[:, :, None, None]


class UNet_conditional(nn.Module):
    def __init__(self, c_in=1, c_out=1, time_dim=512, img_size=512, attn_heads=4,
                 enc_attn=(1, 2, 3, 4, 5, 6), dec_attn=()):
        super().__init__()
        self.time_dim = time_dim
        s = img_size
        enc_ch = [32, 64, 128, 256, 512, 512]               # output channels of the 6 encoder blocks
        enc_size = [s // 2 ** k for k in range(1, 7)]        # 256, 128, 64, 32, 16, 8 for s = 512
        self.inc = DoubleConv(c_in, 16)                      # [CODE-ONLY] not described in the paper
        self.down = nn.ModuleList([Down(ci, co, sz)
                                   for ci, co, sz in zip([16] + enc_ch[:-1], enc_ch, enc_size)])
        self.enc_attn = nn.ModuleDict({str(i): SelfAttention(enc_ch[i - 1], enc_size[i - 1], attn_heads)
                                       for i in enc_attn})
        self.bot = nn.Sequential(DoubleConv(512, 512), DoubleConv(512, 512), DoubleConv(512, 512))  # [CODE-ONLY]
        # decoder blocks named as in the authors' code: 6 = first (deepest), 1 = last (full resolution)
        dec_cfg = {6: (1024, 256, s // 32), 5: (512, 128, s // 16), 4: (256, 64, s // 8),
                   3: (128, 32, s // 4), 2: (64, 16, s // 2), 1: (32, 8, s)}
        self.up = nn.ModuleDict({str(k): Up(*v) for k, v in dec_cfg.items()})
        self.dec_attn = nn.ModuleDict({str(k): SelfAttention(dec_cfg[k][1], dec_cfg[k][2], attn_heads)
                                       for k in dec_attn})
        self.outc = nn.Conv2d(8, c_out, kernel_size=1)

    def pos_encoding(self, t, channels):
        """[PAPER 2.2.3] Transformer positional embedding of the time step (Vaswani et al., ref. [52]):
        sin/cos at geometrically spaced frequencies -> a continuous vector of length `channels`."""
        inv_freq = 1.0 / (10000 ** (torch.arange(0, channels, 2, device=t.device).float() / channels))
        pos_enc_a = torch.sin(t.repeat(1, channels // 2) * inv_freq)
        pos_enc_b = torch.cos(t.repeat(1, channels // 2) * inv_freq)
        return torch.cat([pos_enc_a, pos_enc_b], dim=-1)

    def forward(self, x, t, SHP, Loc, CR, SK, HT, FT, Mag):
        labels = (SHP, Loc, CR, SK, HT, FT, Mag)             # same order as CONDITION_COLUMNS
        t = self.pos_encoding(t.unsqueeze(-1).type(torch.float), self.time_dim)

        skips = [self.inc(x)]                                # skips[k] = output of encoder block k (0 = inc)
        h = skips[0]
        for i, down in enumerate(self.down, start=1):
            h = down(h, t, labels)
            if str(i) in self.enc_attn:
                h = self.enc_attn[str(i)](h)                 # [PAPER 2.2.3] attention after each down-sampling
            skips.append(h)

        h = self.bot(h)

        for k in (6, 5, 4, 3, 2, 1):
            h = self.up[str(k)](h, skips[k - 1], t, labels)  # skip connection from the same-resolution encoder block
            if str(k) in self.dec_attn:
                h = self.dec_attn[str(k)](h)
        return self.outc(h)                                   # predicted noise, same shape as the input image

<div dir="rtl">

## ۴. فرایند انتشار (§2.2.2، معادلهٔ ۱ تا ۶)

**فرایند رفت:** نویز گاوسی به‌تدریج به تصویر اضافه می‌شود. با تعریف $\alpha_t = 1-\beta_t$ و $\bar\alpha_t=\prod_{s\le t}\alpha_s$ می‌توان بدون محاسبهٔ گام‌های میانی مستقیم به گام $t$ رفت:

$$x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\epsilon,\qquad \epsilon\sim\mathcal N(0,\mathbf I)\qquad(\text{معادلهٔ }3)$$

**آموزش:** شبکه نویز افزوده‌شده را برآورد می‌کند و تابع زیان این است (معادلهٔ ۶):

$$L(\theta)=\lVert\epsilon_t-\epsilon_\theta(x_t,t)\rVert_2^2$$

**فرایند بازگشت (نمونه‌برداری):** از نویز خالص $x_T$ شروع می‌کنیم و در هر گام با میانگین تقریبی (معادلهٔ ۵) تصویر را یک گام پاک‌تر می‌کنیم:

$$\tilde\mu_t(x_t,t)\approx\frac{1}{\sqrt{\alpha_t}}\left(x_t-\frac{1-\alpha_t}{\sqrt{1-\bar\alpha_t}}\,\epsilon_\theta(x_t,t)\right)\qquad(\text{معادلهٔ }5)$$

مقاله $\beta_t$ و $T$ را عددی نمی‌دهد؛ مقدارهای $T=1000$ و برنامهٔ خطی $10^{-4}$ تا $0.02$ از کد نویسندگان‌اند. واریانس $\sigma_t^2$ در معادلهٔ ۴ در کد برابر $\beta_t$ است.

</div>

In [ ]:
# ======================================================================
# PAPER MAP | Section 2.2.2, Eq. 1-6 -- forward process, training noise, reverse process (sampling)
# Eq.1  q(x_t|x_{t-1}) = N(sqrt(1-beta_t) x_{t-1}, beta_t I)   -> beta schedule
# Eq.2  alpha_t = 1-beta_t,  alpha_bar_t = prod alpha_s        -> self.alpha, self.alpha_hat
# Eq.3  x_t = sqrt(alpha_bar) x_0 + sqrt(1-alpha_bar) eps       -> noise_images()
# Eq.4-5 reverse step with the approximate mean                 -> sample()
# Eq.6  L = || eps - eps_theta(x_t, t) ||^2                      -> training loop (section 7)
# ======================================================================


class Diffusion:
    def __init__(self, noise_steps=1000, beta_start=1e-4, beta_end=0.02, img_size=512):
        self.noise_steps = noise_steps
        self.beta_start = beta_start
        self.beta_end = beta_end
        self.img_size = img_size

        self.beta = self.prepare_noise_schedule().to(device)   # beta_t        (Eq. 1)
        self.alpha = 1. - self.beta                            # alpha_t       (Eq. 2)
        self.alpha_hat = torch.cumprod(self.alpha, dim=0)      # alpha_bar_t   (Eq. 2)

    def prepare_noise_schedule(self):
        return torch.linspace(self.beta_start, self.beta_end, self.noise_steps)   # [CODE-ONLY] linear schedule

    def noise_images(self, x, t):
        """[PAPER Eq.3] jump straight from the clean image x_0 to the noisy x_t. Returns (x_t, eps)."""
        sqrt_alpha_hat = torch.sqrt(self.alpha_hat[t])[:, None, None, None]
        sqrt_one_minus_alpha_hat = torch.sqrt(1. - self.alpha_hat[t])[:, None, None, None]
        epsilon = torch.randn_like(x)
        return sqrt_alpha_hat * x + sqrt_one_minus_alpha_hat * epsilon, epsilon

    def sample_timesteps(self, n):
        # [CODE-ONLY] t is uniform in {1, ..., T-1}; index 0 of the schedule is never used.
        return torch.randint(low=1, high=self.noise_steps, size=(n,))

    def sample(self, model, n, SHP, Loc, CR, SK, HT, FT, Mag):
        """[PAPER 2.2.2, Eq.4-5] generate n images: start from pure Gaussian noise x_T and denoise step by
        step with the trained network, conditioned on the 7 process parameters.
        [FIX] the original had a classifier-free-guidance branch that crashed (the model has no 'null'
        condition) and the paper does not use guidance, so it was removed."""
        was_training = model.training
        model.eval()
        amp_enabled = torch.cuda.is_available()   # [ENGINEERING] fp16 on GPU
        with torch.no_grad(), torch.amp.autocast(device.type, enabled=amp_enabled):
            x = torch.randn((n, 1, self.img_size, self.img_size)).to(device)        # x_T ~ N(0, I)
            for i in reversed(range(1, self.noise_steps)):
                t = (torch.ones(n) * i).long().to(device)
                predicted_noise = model(x, t, SHP, Loc, CR, SK, HT, FT, Mag)         # eps_theta(x_t, t)
                alpha = self.alpha[t][:, None, None, None]
                alpha_hat = self.alpha_hat[t][:, None, None, None]
                beta = self.beta[t][:, None, None, None]
                noise = torch.randn_like(x) if i > 1 else torch.zeros_like(x)        # no noise on the last step
                # Eq.5 mean  +  sigma_t * z   with sigma_t^2 = beta_t  [CODE-ONLY choice of sigma_t]
                x = 1 / torch.sqrt(alpha) * (x - ((1 - alpha) / torch.sqrt(1 - alpha_hat)) * predicted_noise) \
                    + torch.sqrt(beta) * noise
        model.train(was_training)                  # [FIX] restore the mode the model was in
        return x.float()

<div dir="rtl">

## ۵. داده (§2.2.1 و §2.2.4)

**ساختار دادهٔ لازم** (همان نوتبوک اصلی): در یک پوشه زیر My Drive

</div>

```
<root>/Training/<series>/*.jpg      e.g.: Training/CM01-0500/*.jpg
<root>/Testing/<series>/*.jpg       e.g.: Testing/CM06-0500/*.jpg
<root>/Training Labels.xlsx
<root>/Testing Labels.xlsx
```

<div dir="rtl">

هر «series» یک پوشه از تصاویر یک نمونه با یک بزرگ‌نمایی است، یعنی یک **کلاس تصویری** در اصطلاح مقاله. نام آن چنین خوانده می‌شود: حرف اول هندسهٔ شمش (C استوانه‌ای، P پیش‌فرم)، حرف دوم محل نمونه (M جان، S بال کوتاه، L بال بلند)، دو رقم بعدی شمارهٔ قطعه و بعد از خط تیره بزرگ‌نمایی.

**تفکیک آموزش و آزمون (§2.2.4، leave-one-category-out):** از ۱۳۱ کلاس، ۱۷ کلاس (حدود ۱۳٪) برای آزمون کنار گذاشته شده‌اند و ۱۱۴ کلاس (۸۷٪) آموزشی‌اند. این دقیقاً همان تفکیک پوشه‌های `Training/` و `Testing/` است؛ پس تقسیم تصادفی جدیدی لازم نیست و شرایط آزمون واقعاً دیده‌نشده‌اند. ۱۷ کلاس آزمون سه سناریو دارند: خروج یک قطعهٔ کامل، خروج یک نمونهٔ متالوگرافی و خروج یک بزرگ‌نمایی.

**پیش‌پردازش (§2.2.1)** در `AZ80SeriesDataset.__getitem__` اتفاق می‌افتد و هر بار که تصویر خوانده می‌شود اجرا می‌شود:

1. تبدیل به خاکستری و برش تصادفی `IMAGE_SIZE × IMAGE_SIZE` از تصویر ۱۰۲۴×۷۶۸،
2. نرمال‌سازی شدت پیکسل‌ها از $[0,255]$ به $[-1,1]$،
3. قرینه‌سازی افقی و عمودی تصادفی با احتمال ۰٫۵ (فقط برای داده‌های آموزشی؛ ریزساختار به جهت‌گیری حساس نیست).

**کدگذاری برچسب‌ها:** هر ستون فایل برچسب یک کد صحیح دارد. معنی کدها در سلول تنظیمات (`COND_VOCAB`) آمده است. سلول زیر پس از خواندن برچسب‌ها بررسی می‌کند که هر کد در بازهٔ مجاز باشد، چون کد خارج از بازه در `nn.Embedding` خطای مبهمی می‌دهد.

**DataLoader:** با `drop_last=True` و shuffle ساخته می‌شود و در حلقهٔ آموزش در **هر epoch از نو** اجرا می‌شود (اصلاح باگ `itertools.cycle`).

</div>

In [ ]:
# ======================================================================
# PAPER MAP | Section 2.2.1 (data preprocessing) and 2.2.4 (leave-one-category-out split)
# 114 training classes / 17 testing classes = the Training/ and Testing/ folders.
# Image -> grayscale -> random crop -> [-1, 1] -> random flips (p = 0.5).
# ======================================================================

# --- Data source resolution -------------------------------------------------
# Rather than require an exact Drive path, search structurally under My
# Drive for any folder that actually looks like this dataset (has Training/
# and Testing/ subfolders), a couple of levels deep, plus a local `data/`
# fallback for running off a repo clone outside Colab. Put the dataset
# anywhere under My Drive -- it doesn't matter what you name the folder.
DRIVE_ROOT = Path('/content/drive/MyDrive')
DRIVE_SEARCH_DEPTH = 2  # how many folder levels under My Drive to search --
                        # kept shallow since a personal Drive can have many
                        # unrelated folders and Drive-mounted I/O is slow;
                        # raise it if your dataset is nested deeper.


def _looks_like_dataset_root(c: Path) -> bool:
    return (
        c.is_dir()
        and (c / 'Training').is_dir()
        and (c / 'Testing').is_dir()
        and (c / 'Training Labels.xlsx').is_file()
        and (c / 'Testing Labels.xlsx').is_file()
    )


def resolve_data_root() -> Path:
    candidates = []
    if DRIVE_ROOT.exists():
        level = [DRIVE_ROOT]
        for _ in range(DRIVE_SEARCH_DEPTH):
            next_level = []
            for d in level:
                if d.is_dir():
                    next_level.extend(p for p in d.iterdir() if p.is_dir())
            candidates.extend(next_level)
            level = next_level
    candidates += [Path('data')]  # local clone fallback (outside Colab)

    # Require Training/, Testing/, and both *Labels.xlsx to all be present --
    # matching on Training/+Testing/ alone can grab an unrelated or
    # incomplete folder that happens to share those subfolder names (e.g. a
    # stray/partial upload sitting elsewhere in the same Drive).
    matches = [c for c in candidates if _looks_like_dataset_root(c)]
    if not matches:
        # Fall back to the looser check only to give a more informative
        # error message about what's actually missing, if anything close
        # was found.
        near_matches = [c for c in candidates if c.is_dir() and (c / 'Training').is_dir() and (c / 'Testing').is_dir()]
        if near_matches:
            missing = [
                f"{c} (missing: {', '.join(n for n in ['Training Labels.xlsx', 'Testing Labels.xlsx'] if not (c / n).is_file())})"
                for c in near_matches
            ]
            raise FileNotFoundError(
                "Found folder(s) with Training/ and Testing/ but missing the label "
                f"xlsx files: {missing}. Fix the folder's contents, or move/rename "
                "the incomplete one so it isn't found first."
            )
        raise FileNotFoundError(
            "Could not find a data root. Upload a dataset folder to Google Drive "
            "that mirrors data/'s layout (Training/, Testing/, *Labels.xlsx) "
            "somewhere under My Drive, or run this notebook from a clone of the "
            "repo with a 'data/' folder present at the root."
        )
    if len(matches) > 1:
        print(f'Warning: multiple candidate dataset folders found: {matches}. Using the first: {matches[0]}')
    print(f'Using data root: {matches[0].resolve()}')
    return matches[0]


DATA_ROOT = resolve_data_root()
TRAIN_DIR = DATA_ROOT / 'Training'
TEST_DIR = DATA_ROOT / 'Testing'
TRAIN_LABELS_XLSX = DATA_ROOT / 'Training Labels.xlsx'
TEST_LABELS_XLSX = DATA_ROOT / 'Testing Labels.xlsx'

# CONDITION_COLUMNS and COND_VOCAB are defined in the configuration cell (section 2).


def load_label_table(xlsx_path: Path) -> pd.DataFrame:
    """Parse a `<Training|Testing> Labels.xlsx` sheet (one row per field, one
    column per series -- see the Data Loading markdown above) into a
    DataFrame indexed by series name, e.g. table.loc['CM01-0500'].shape."""
    raw = pd.read_excel(xlsx_path, header=None).set_index(0)
    table = raw.T.reset_index(drop=True)
    table = table.rename(columns={
        'Label': 'series', 'Class': 'class_id', 'Shape': 'shape',
        'Location': 'location', 'Cooling rate': 'cooling_rate',
        'Soaking': 'soaking', 'Heat-treatment': 'heat_treatment',
        'Forging Temp': 'forging_temp', 'Magnification': 'magnification',
    })
    int_cols = ['class_id'] + CONDITION_COLUMNS
    table[int_cols] = table[int_cols].astype(int)
    return table.set_index('series')


class AZ80SeriesDataset(Dataset):
    """One sample = one SEM image + the 7 process-parameter labels of the
    series (folder) it belongs to. Series folders hold a variable number of
    images (a handful up to ~9 in this dataset)."""

    IMG_EXTS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}

    def __init__(self, image_dir: Path, label_table: pd.DataFrame, crop_size: int = 512, augment: bool = False):
        self.label_table = label_table
        self.samples = []  # (path, series) pairs
        for series_dir in sorted(Path(image_dir).iterdir()):
            if not series_dir.is_dir() or series_dir.name not in label_table.index:
                continue
            for f in sorted(series_dir.iterdir()):
                if f.suffix.lower() in self.IMG_EXTS:
                    self.samples.append((f, series_dir.name))
        if not self.samples:
            raise RuntimeError(f'No labeled images found under {image_dir}')

        # Section 2.2.1: images are cropped to 512x512 "at the beginning of
        # each iteration" and normalized to [-1, 1] (matches whole_transform
        # in ddpm.py). pad_if_needed is a defensive addition not present in
        # the original code, in case a source image is smaller than 512px.
        self.base_transform = transforms.Compose([
            transforms.Grayscale(),
            transforms.RandomCrop(crop_size, pad_if_needed=True, padding_mode='reflect'),
            transforms.ToTensor(),
            transforms.Lambda(lambda t: (t * 2) - 1),
        ])
        # aug_transform in ddpm.py -- horizontal/vertical flip, p=0.5 each.
        self.aug_transform = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
        ]) if augment else None

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, series = self.samples[idx]
        image = Image.open(path).convert('RGB')
        image = self.base_transform(image)
        if self.aug_transform is not None:
            image = self.aug_transform(image)
        row = self.label_table.loc[series]
        cond = torch.tensor([int(row[c]) for c in CONDITION_COLUMNS], dtype=torch.long)
        return image, cond, series


train_labels = load_label_table(TRAIN_LABELS_XLSX)
test_labels = load_label_table(TEST_LABELS_XLSX)
def validate_labels(table, name):
    """[FIX] an out-of-range code would crash nn.Embedding with an unhelpful error."""
    for c in CONDITION_COLUMNS:
        bad = table[~table[c].between(0, COND_VOCAB[c] - 1)]
        assert bad.empty, f"{name}: column '{c}' has codes outside 0..{COND_VOCAB[c] - 1}: {sorted(set(bad[c]))}"


validate_labels(train_labels, 'Training Labels')
validate_labels(test_labels, 'Testing Labels')
n_total = len(train_labels) + len(test_labels)
print(f'{len(train_labels)} training series, {len(test_labels)} testing series '
      f'({len(train_labels) / n_total:.1%} / {len(test_labels) / n_total:.1%} split)')

train_dataset = AZ80SeriesDataset(TRAIN_DIR, train_labels, crop_size=IMAGE_SIZE, augment=True)
test_dataset = AZ80SeriesDataset(TEST_DIR, test_labels, crop_size=IMAGE_SIZE, augment=False)
print(f'{len(train_dataset)} training images, {len(test_dataset)} testing images')
print(f'batch size {BATCH_SIZE} -> {len(train_dataset) // BATCH_SIZE} optimiser steps per epoch')

# [FIX] the training loop iterates this loader afresh in every epoch (see section 7), so the random
# crop / flips / shuffle change every time. persistent_workers avoids re-spawning workers each epoch.
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
                          drop_last=True, persistent_workers=True, pin_memory=torch.cuda.is_available())

<div dir="rtl">

## ۶. پیکربندی آموزش (§2.2.3)

- **بهینه‌ساز و زیان:** Adam و MSE بین نویز واقعی و نویز برآوردشده (معادلهٔ ۶).
- **EMA:** نسخهٔ میانگین‌گرفته‌شدهٔ وزن‌ها (`ema_model`) برای نمونه‌برداری و محاسبهٔ FID به کار می‌رود و آموزش را پایدار می‌کند.
- **checkpoint:** در تکرارهای ۳۰ و هر ۶۰۰ تکرار ذخیره می‌شود تا بتوان شکل ۲ مقاله را بازسازی کرد. checkpoint کامل (مدل، EMA و Adam) حدود ۵٫۶ گیگابایت است و فقط جدیدترین آن کامل می‌ماند؛ قدیمی‌ها فقط وزن EMA (حدود ۱٫۴ گیگابایت) را نگه می‌دارند.
- **اصلاح ترتیب ذخیره:** اول checkpoint جدید نوشته می‌شود و بعد قدیمی‌ها کم‌حجم می‌شوند. بدین ترتیب اگر نوشتن شکست بخورد (قطع شدن Colab یا پر شدن Drive) checkpoint قبلی سالم می‌ماند. هزینهٔ آن این است که لحظه‌ای دو checkpoint کامل کنار هم روی Drive هستند؛ اگر فضایت کم است `SAFE_CHECKPOINT_ORDER = False` بگذار.
- **resume خودکار:** `RESUME_FROM = 'auto'` جدیدترین checkpoint **کامل** را پیدا می‌کند. اگر یکای تکرار (`epoch` یا `step`) با آن checkpoint فرق کند خطا می‌دهد.

</div>

In [ ]:
# ======================================================================
# PAPER MAP | Section 2.2.3 -- Adam optimiser, MSE loss (Eq.6), EMA; checkpointing and resume
# The model, optimiser, EMA copy and diffusion object are created here.
# ======================================================================


import gc
import os
import re
import shutil

# ---- Output folders (smoke-test runs get their own folder so they can never be auto-resumed) ------
OUTPUT_SUBDIR = 'az80_ddpm_outputs_smoke' if SMOKE_TEST else 'az80_ddpm_outputs'
DRIVE_OUTPUT_DIR = (DATA_ROOT.parent / OUTPUT_SUBDIR) if IN_COLAB else (Path('.') / OUTPUT_SUBDIR)
CHECKPOINT_DIR = DRIVE_OUTPUT_DIR / 'checkpoints'
MODELS_DIR = DRIVE_OUTPUT_DIR / 'models'
GENERATED_DIR = DRIVE_OUTPUT_DIR / 'generated_samples'
for _d in (CHECKPOINT_DIR, MODELS_DIR, GENERATED_DIR):
    _d.mkdir(parents=True, exist_ok=True)
print(f'Checkpoints/models/samples will be written under: {DRIVE_OUTPUT_DIR.resolve()}')

RESUME_FROM = 'auto'   # 'auto' = newest FULL checkpoint in CHECKPOINT_DIR | None = start from scratch | a Path


# ---- Checkpoint helpers ------------------------------------------------------------------------
def load_ckpt_cpu(path):
    """Read a checkpoint to the CPU (lazily if possible) -- never straight onto the GPU."""
    try:
        return torch.load(path, map_location='cpu', mmap=True)
    except (TypeError, RuntimeError, ValueError):
        return torch.load(path, map_location='cpu')


def checkpoint_iteration(path):
    m = re.search(r'iter(\d+)', Path(path).name)
    return int(m.group(1)) if m else None


def find_latest_checkpoint(dir_):
    """[FIX] newest checkpoint that still holds the training state (older ones are slimmed to EMA-only)."""
    for p in sorted(dir_.glob('ckpt_iter*.pth.tar'), reverse=True):
        try:
            ckpt = load_ckpt_cpu(p)
            full = 'model_state' in ckpt
            del ckpt
            if full:
                return p
        except Exception as e:   # unreadable / half-written file
            print(f'Skipping unreadable checkpoint {p.name}: {e}')
    return None


def slim_checkpoint(path):
    """Rewrite a full checkpoint (model + EMA + Adam) as EMA-only (about 4x smaller).
    The EMA weights are all that sampling, FID and the paper figures need."""
    path = Path(path)
    ckpt = load_ckpt_cpu(path)
    if 'model_state' not in ckpt:
        return False
    slim = {'iteration': ckpt.get('iteration', checkpoint_iteration(path)),
            'iteration_unit': ckpt.get('iteration_unit', ITERATION_UNIT),
            'ema_model_state': ckpt['ema_model_state']}
    tmp = path.with_name(path.name + '.tmp')
    try:
        torch.save(slim, tmp)
        del ckpt, slim
        gc.collect()
        os.replace(tmp, path)
    finally:
        tmp.unlink(missing_ok=True)
    return True


def slim_old_checkpoints(before_iteration):
    for p in sorted(CHECKPOINT_DIR.glob('ckpt_iter*.pth.tar')):
        it = checkpoint_iteration(p)
        if it is not None and it < before_iteration and slim_checkpoint(p):
            print(f'Slimmed {p.name} to EMA-only weights')


def safe_slim(iteration):
    """[FIX] slimming old checkpoints is only a disk-saving step; a failure must never stop training."""
    try:
        slim_old_checkpoints(iteration)
    except (OSError, RuntimeError) as e:
        print(f'WARNING: could not slim older checkpoints ({e}); they stay full-size.')


def save_checkpoint(path, iteration, slim_old=True):
    """Save a full checkpoint atomically. Returns True on success.
    [FIX] With SAFE_CHECKPOINT_ORDER the new file is written BEFORE the old ones are slimmed, and a failed
    write (disk full, Drive disconnect) only prints a warning -- the previous checkpoint stays intact."""
    path = Path(path)
    if slim_old and not SAFE_CHECKPOINT_ORDER:
        safe_slim(iteration)
    tmp = path.with_name(path.name + '.tmp')
    try:
        torch.save({
            'iteration': iteration,
            'iteration_unit': ITERATION_UNIT,
            'model_state': model.state_dict(),
            'ema_model_state': ema_model.state_dict(),
            'model_optimizer': optimizer.state_dict(),
            'ema_step': ema.step,
        }, tmp)
        os.replace(tmp, path)
    except (OSError, RuntimeError) as e:
        print(f'WARNING: could not write {path.name}: {e}\n'
              '         Earlier checkpoints are untouched. Free some disk space or set SAFE_CHECKPOINT_ORDER = False.')
        return False
    finally:
        tmp.unlink(missing_ok=True)
    if slim_old and SAFE_CHECKPOINT_ORDER:
        safe_slim(iteration)
    return True


if RESUME_FROM == 'auto':
    RESUME_FROM = find_latest_checkpoint(CHECKPOINT_DIR)
    print(f'Auto-resume found: {RESUME_FROM}' if RESUME_FROM else 'Auto-resume: no checkpoint found, starting fresh.')

# ---- Model, optimiser, EMA -----------------------------------------------------------------------
model = UNet_conditional(img_size=IMAGE_SIZE, attn_heads=ATTN_HEADS,
                         enc_attn=ATTN_ENCODER_BLOCKS, dec_attn=ATTN_DECODER_BLOCKS).to(device)
print(f'U-Net parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M')
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)   # [PAPER 2.2.3] Adam
mse = nn.MSELoss()                                              # [PAPER Eq.6] mean squared error
diffusion = Diffusion(noise_steps=NOISE_STEPS, beta_start=BETA_START, beta_end=BETA_END, img_size=IMAGE_SIZE)
ema = EMA(EMA_BETA)                                             # [PAPER 2.2.3] EMA of the weights
ema_model = copy.deepcopy(model).eval().requires_grad_(False)

# [ENGINEERING] mixed precision: the same maths in fp16 where safe, about half the memory.
AMP_ENABLED = AMP_REQUESTED and torch.cuda.is_available()
scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)

# ---- Resume ----------------------------------------------------------------------------------------
start_iteration = 1
if RESUME_FROM is not None and Path(RESUME_FROM).exists():
    ckpt = load_ckpt_cpu(RESUME_FROM)
    if 'model_state' not in ckpt:
        raise RuntimeError(f'{RESUME_FROM} holds only EMA weights (older checkpoints are slimmed to save disk) '
                           '-- resume from the NEWEST checkpoint instead.')
    if ckpt.get('iteration_unit', ITERATION_UNIT) != ITERATION_UNIT:
        raise RuntimeError(f"checkpoint counts in '{ckpt.get('iteration_unit')}' but ITERATION_UNIT is "
                           f"'{ITERATION_UNIT}'; change the setting or start from scratch.")
    model.load_state_dict(ckpt['model_state'])
    ema_model.load_state_dict(ckpt['ema_model_state'])
    optimizer.load_state_dict(ckpt['model_optimizer'])
    ema.step = ckpt.get('ema_step', 0)
    start_iteration = ckpt.get('iteration', 0) + 1
    del ckpt
    gc.collect()
    print(f'Resumed from {RESUME_FROM} at {ITERATION_UNIT} {start_iteration}')
else:
    print('Training from randomly initialized weights.')

<div dir="rtl">

## ۷. حلقهٔ آموزش (معادلهٔ ۶)

در هر گام: یک دسته تصویر تمیز $x_0$ ← یک $t$ تصادفی برای هر تصویر ← ساخت $x_t$ با معادلهٔ ۳ ← برآورد نویز با U-Net ← زیان MSE ← گام Adam ← به‌روزرسانی EMA.

**«iteration» در این نوتبوک** یعنی یک epoch (پیش‌فرض)، یعنی یک دور کامل DataLoader. سه‌شش‌صدمین تکرار شکل ۲ مقاله با epoch ۶۰۰ هم‌خوان است.

**اصلاح باگ:** نوتبوک اصلی از `itertools.cycle(train_loader)` استفاده می‌کرد. این تابع دسته‌های دور اول را ذخیره می‌کند و از دور دوم فقط همان‌ها را تکرار می‌کند؛ DataLoader دیگر اجرا نمی‌شد و برش و قرینه‌سازی تصادفی (که مقاله در هر تکرار می‌گوید) عملاً فقط یک بار اعمال می‌شد. اینجا DataLoader در هر epoch از نو اجرا می‌شود.

پس از هر تکرار (epoch) سه کار ممکن است انجام شود: چاپ میانگین زیان، تولید چند تصویر پیشرفت روی شرایط دیده‌نشده (با وزن‌های EMA) و ذخیرهٔ checkpoint.

</div>

In [ ]:
# ======================================================================
# PAPER MAP | Section 2.2.2-2.2.3 -- sampling helper used for progress images
# Samples from the EMA model on a fixed set of held-out (unseen) conditions, like Fig. 2.
# ======================================================================

def sample_and_show(source_model, conditions, title_prefix='', save_path=None):
    """conditions: list of dicts with the 7 CONDITION_COLUMNS keys (already
    integer-encoded) -- generates and displays one image per condition."""
    n = len(conditions)
    cond_tensor = torch.tensor([[c[k] for k in CONDITION_COLUMNS] for c in conditions],
                                dtype=torch.long, device=device)
    shp, loc, cr, sk, ht, ft, mag = cond_tensor.unbind(dim=1)
    images = diffusion.sample(source_model, n, shp, loc, cr, sk, ht, ft, mag)
    images = (images.clamp(-1, 1) + 1) / 2  # [-1,1] -> [0,1] for display

    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    axes = np.atleast_1d(axes)
    for ax, img in zip(axes, images):
        ax.imshow(img.squeeze(0).cpu(), cmap='gray')
        ax.axis('off')
    fig.suptitle(title_prefix)
    if save_path:
        fig.savefig(save_path, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    return images


# A handful of fixed held-out (unseen) conditions to visualize training
# progress on, chosen once so the same conditions recur across the whole run.
_progress_rng = np.random.RandomState(SEED)
_progress_idx = _progress_rng.choice(len(test_labels), size=min(4, len(test_labels)), replace=False)
progress_conditions = [
    {k: int(test_labels.iloc[i][k]) for k in CONDITION_COLUMNS} for i in _progress_idx
]

In [ ]:
# ======================================================================
# PAPER MAP | Section 2.2.3, Eq.6 -- training loop
# [FIX] the DataLoader is re-iterated every epoch (the original itertools.cycle replayed epoch 1).
# One 'iteration' = one epoch when ITERATION_UNIT == 'epoch' (see section 2).
# ======================================================================


def train_step(images, cond):
    """One optimiser step: Eq.3 (add noise) -> network -> Eq.6 (MSE) -> Adam -> EMA."""
    images = images.to(device, non_blocking=True)
    shp, loc, cr, sk, ht, ft, mag = cond.to(device).unbind(dim=1)
    t = diffusion.sample_timesteps(images.shape[0]).to(device)    # a random time step per image
    x_t, noise = diffusion.noise_images(images, t)                # [PAPER Eq.3]

    optimizer.zero_grad(set_to_none=True)
    with torch.amp.autocast(device.type, enabled=AMP_ENABLED):
        predicted_noise = model(x_t, t, shp, loc, cr, sk, ht, ft, mag)   # eps_theta(x_t, t, conditions)
        loss = mse(noise, predicted_noise)                                # [PAPER Eq.6]

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    ema.step_ema(ema_model, model, step_start_ema=EMA_STEP_START)         # [PAPER 2.2.3] EMA
    return loss.item()


loss_history = []   # mean loss per iteration (epoch, or single step in 'step' mode)
start_time = time.time()


def end_of_iteration(iteration, mean_loss):
    loss_history.append(mean_loss)
    if iteration % LOG_EVERY == 0:
        elapsed_min = (time.time() - start_time) / 60
        print(f'[{ITERATION_UNIT} {iteration}/{NUM_ITERATIONS}] '
              f'loss={np.mean(loss_history[-LOG_EVERY:]):.4f} elapsed={elapsed_min:.1f} min')
    if iteration % SAMPLE_EVERY == 0 or iteration == NUM_ITERATIONS:
        sample_and_show(ema_model, progress_conditions, title_prefix=f'EMA samples @ {ITERATION_UNIT} {iteration}',
                        save_path=GENERATED_DIR / f'progress_iter{iteration:06d}.png')
    if iteration % CHECKPOINT_EVERY == 0 or iteration == NUM_ITERATIONS or iteration in EXTRA_CHECKPOINT_ITERS:
        ckpt_path = CHECKPOINT_DIR / f'ckpt_iter{iteration:06d}.pth.tar'
        if save_checkpoint(ckpt_path, iteration):
            print(f'Saved checkpoint: {ckpt_path}')


def infinite_batches(loader):
    """[FIX] replaces itertools.cycle: re-run the DataLoader (new shuffle, crops, flips) when it is exhausted."""
    while True:
        for batch in loader:
            yield batch


model.train()
if start_iteration > NUM_ITERATIONS:
    print(f'Nothing to train: already at {ITERATION_UNIT} {start_iteration - 1} of {NUM_ITERATIONS}.')
elif ITERATION_UNIT == 'epoch':
    pbar = tqdm(range(start_iteration, NUM_ITERATIONS + 1), initial=start_iteration - 1,
                total=NUM_ITERATIONS, desc='epochs')
    for iteration in pbar:
        epoch_losses = []
        for images, cond, _ in train_loader:         # fresh pass over the data every epoch
            epoch_losses.append(train_step(images, cond))
        pbar.set_postfix(loss=float(np.mean(epoch_losses)))
        end_of_iteration(iteration, float(np.mean(epoch_losses)))
else:   # ITERATION_UNIT == 'step'
    batches = infinite_batches(train_loader)
    pbar = tqdm(range(start_iteration, NUM_ITERATIONS + 1), initial=start_iteration - 1,
                total=NUM_ITERATIONS, desc='steps')
    for iteration in pbar:
        images, cond, _ = next(batches)
        step_loss = train_step(images, cond)
        pbar.set_postfix(loss=step_loss)
        end_of_iteration(iteration, step_loss)

print(f'Training finished at {ITERATION_UNIT} {NUM_ITERATIONS}.')
if torch.cuda.is_available():
    print(torch.cuda.memory_summary(device=device))

<div dir="rtl">

## ۸. نمونه‌برداری و ارزیابی بصری (§3.1)

با مدل EMA برای هر ترکیب دلخواه از پارامترهای فرایند تصویر می‌سازیم و یک تصویر مصنوعی را با تصویر واقعی همان series مقایسه می‌کنیم؛ یک بار برای شرایط **دیده‌نشده** (از `Testing/`) و یک بار برای شرایط **دیده‌شده** (از `Training/`). این همان مقایسهٔ شکل‌های ۳ و ۴ مقاله است. این ارزیابی فقط کیفی است؛ ارزیابی کمی مقاله روی سه شاخص ریخت‌شناسی (شکل ۵) در این نوتبوک نیست.

</div>

In [ ]:
# ======================================================================
# PAPER MAP | Section 3.1 / Fig. 3-4 -- one synthesized vs. real image, unseen then seen condition

# ======================================================================

def compare_real_vs_synthesized(dataset: AZ80SeriesDataset, series_name: str = None, source_model=None):
    """dataset: train_dataset or test_dataset. series_name defaults to the
    first series that actually has images loaded in `dataset` -- a labels
    xlsx can list a series that has no corresponding uploaded images (e.g. a
    partially-populated dataset), so we pick from dataset.samples (built
    from what's really on disk) rather than dataset.label_table.index
    (built from the xlsx, which may list more series than were uploaded)."""
    source_model = source_model if source_model is not None else ema_model
    if series_name is None:
        series_name = dataset.samples[0][1]
    real_path = next((p for p, s in dataset.samples if s == series_name), None)
    if real_path is None:
        raise ValueError(
            f"No images found for series '{series_name}' in this dataset -- it "
            "may be listed in the labels xlsx but missing from the uploaded "
            "image folders."
        )
    real_image = transforms.Compose([
        transforms.Grayscale(),
        transforms.RandomCrop(IMAGE_SIZE, pad_if_needed=True, padding_mode='reflect'),
    ])(Image.open(real_path).convert('RGB'))

    row = dataset.label_table.loc[series_name]
    cond = {k: int(row[k]) for k in CONDITION_COLUMNS}
    synth = sample_and_show(source_model, [cond], title_prefix=f'synthesized: {series_name}')

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(real_image, cmap='gray')
    axes[0].set_title(f'real: {series_name}')
    axes[0].axis('off')
    axes[1].imshow(synth[0].squeeze(0).cpu(), cmap='gray')
    axes[1].set_title('synthesized')
    axes[1].axis('off')
    plt.show()


# Example: one unseen (held-out) condition, one seen (training) condition.
compare_real_vs_synthesized(test_dataset)
compare_real_vs_synthesized(train_dataset)

<div dir="rtl">

## ۹. ذخیرهٔ نهایی

مدل کاملاً آموزش‌دیده در `models/` ذخیره می‌شود. آخرین checkpoint خودش همان مدل نهایی است؛ برای صرفه‌جویی در فضا، اگر سیستم‌فایل hard link بپذیرد فقط یک پیوند ساخته می‌شود و در Drive (که hard link ندارد) نسخهٔ دوم نوشته نمی‌شود.

</div>

In [ ]:
# ======================================================================
# PAPER MAP | (no paper section) -- final model file

# ======================================================================

import os

final_model_path = MODELS_DIR / 'az80_ddpm_final.pth.tar'
last_ckpt = CHECKPOINT_DIR / f'ckpt_iter{NUM_ITERATIONS:06d}.pth.tar'
final_model_path.unlink(missing_ok=True)
if last_ckpt.exists():
    # The last checkpoint IS the final model. Hard-link it instead of writing a second
    # ~5.6 GiB copy, which would not fit in Kaggle's 20 GiB or a 15 GB Google Drive.
    try:
        os.link(last_ckpt, final_model_path)
        print(f'Final model (model + EMA + optimizer state): {final_model_path}  (hard link to {last_ckpt.name})')
    except OSError:
        # Some filesystems (e.g. Google Drive) have no hard links. Don't duplicate 5.6 GiB:
        # the last checkpoint already is the final model, in the same format.
        print(f'This filesystem has no hard links, so no second copy was written.\n'
              f'The final model (model + EMA + optimizer state) is {last_ckpt}')
else:
    if save_checkpoint(final_model_path, NUM_ITERATIONS, slim_old=False):
        print(f'Final model + EMA model + optimizer state saved to {final_model_path}')
    else:
        print('The final model could NOT be written (see the warning above).')

<div dir="rtl">

## ۱۰. بازسازی شکل‌های مقاله (شکل ۲، ۳ و ۴)

سه بخش زیر شکل‌های نتایج مقاله را از مدلِ **خودت** دوباره می‌کشند.

| بخش | شکل مقاله | محتوا | نیاز |
|---|---|---|---|
| ۱۰-۱ | شکل ۲ | منحنی FID بر حسب تکرار و تحول یک تصویر از نویز تا تصویر نهایی | checkpointها، `pytorch-fid` |
| ۱۰-۲ | شکل ۳ | ۸ شرط دیده‌شده؛ تصویر مصنوعی و واقعی کنار هم با پارامترها زیر هر جفت | مدل آموزش‌دیده |
| ۱۰-۳ | شکل ۴ | شرایط دیده‌نشده: خروج یک قطعه، یک نمونه و یک بزرگ‌نمایی | مدل آموزش‌دیده |

شکل‌ها در `GENERATED_DIR/paper_figures/` ذخیره می‌شوند. تولید هر تصویر حدود $T$ بار اجرای U-Net است، پس این بخش کندترین بخش نوتبوک است؛ تصاویر در `paper_figures/cache/` ذخیره می‌شوند تا اجرای مجدد از اول شروع نشود. **[FIX]** کلید cache اکنون اثرانگشت وزن‌های مدل را هم دارد تا بعد از آموزش مجدد تصاویر مدل قبلی استفاده نشوند.

**حافظه:** بعد از آموزش، وزن‌های آموزشی مدل و حالت Adam از GPU آزاد می‌شوند و فقط `ema_model` می‌ماند (`RELEASE_TRAINING_MEMORY`). در نتیجه **اجرای دوبارهٔ حلقهٔ آموزش در همان جلسه ممکن نیست**؛ برای ادامهٔ آموزش، سلول‌های بخش ۶ را با `RESUME_FROM` از نو اجرا کن.

**احتیاط‌ها:**

- **مقدارهای FID:** مقاله FID را روی مجموعه‌های ۲۰۰ تصویری حساب کرده است. اینجا پیش‌فرض ۱۶ تصویر است، پس FID به سمت بالا اریب است؛ شکل منحنی و فاصله‌اش تا خط مبنا را مقایسه کن، نه عدد مطلق را.
- **رنگ فازها:** در مقاله برچسب‌های رنگی Mg₁₇Al₁₂ (A یوتکتیک، B بین‌دانه‌ای، C کروی، D پیوسته) دستی گذاشته شده‌اند. اینجا با یک قاعدهٔ ساده روی روشنایی و شکل ذره، به‌طور یکسان برای تصاویر واقعی و مصنوعی، مشخص می‌شوند و تقریبی‌اند (`HIGHLIGHT_PHASES = False` تصویر ساده می‌دهد).
- **مقیاس:** تصاویر دادهٔ آموزشی اطلاعات مقیاس ندارند، پس خط مقیاس کشیده نمی‌شود مگر `SCALE_BARS` را پر کنی.
- **برچسب خیساندن:** عبارت‌هایی مثل «1.5h*» و «2h†» از کدهای فایل برچسب ساخته می‌شوند، با همان پانویس مقاله.

</div>

In [ ]:
# ======================================================================
# PAPER MAP | Figures 2-4 -- shared helpers (image generation + cache, phase highlighting, drawing)
# Everything the three figure sections below build on.
# ======================================================================

# --- Shared helpers for the paper-figure sections (9.1 - 9.3) ---------------
# These cells build on the notebook's earlier sections. After a runtime restart
# (or a fresh Colab/Kaggle session) run Sections 1-5 first -- "Run all" also
# works: with RESUME_FROM at your last checkpoint the training loop is a no-op.
_needed = ['device', 'SEED', 'IMAGE_SIZE', 'CONDITION_COLUMNS', 'train_dataset', 'test_dataset',
           'train_labels', 'test_labels', 'AZ80SeriesDataset', 'model', 'optimizer', 'ema_model',
           'diffusion', 'start_iteration', 'GENERATED_DIR', 'CHECKPOINT_DIR']
_missing = [n for n in _needed if n not in globals()]
if _missing:
    raise RuntimeError(
        f'Section 9 needs the earlier sections to have run in this session; missing: {_missing}. '
        'Run Sections 1-5 first (Setup, Model, Diffusion, Data Loading, Training Configuration).')
del _needed, _missing

import contextlib
import gc
import hashlib
import json
import re

import matplotlib.patheffects as pe
from matplotlib.patches import FancyArrowPatch, Patch, Rectangle
from scipy import ndimage as ndi
from scipy.spatial import cKDTree
from skimage import measure

PAPER_FIG_DIR = GENERATED_DIR / 'paper_figures'
PAPER_CACHE_DIR = PAPER_FIG_DIR / 'cache'
PAPER_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Which weights the figures use. None -> the in-memory EMA model (i.e. the end
# of the training loop above). Or point at a checkpoint, e.g.
# CHECKPOINT_DIR / 'ckpt_iter003600.pth.tar', to plot from a saved run.
FIGURE_MODEL_CHECKPOINT = None

GEN_BATCH_SIZE = 4  # images generated per sampling call (memory-bound, like BATCH_SIZE)

# Human-readable names for the integer condition codes. The order matches the
# vocabularies in the model cell (e.g. cooling_rate 0/1/2 = 1.5/6/10.4 C/s).
COND_TEXT = {
    'shape': ('Cast geometry', ['Cylinder', 'Preform']),
    'location': ('Location', ['Tall flange', 'Web', 'Short flange']),
    'cooling_rate': ('Cooling rate', ['1.5°C/s', '6.0°C/s', '10.4°C/s']),
    'soaking': ('Soaking process', ['Normal', '1.5h*', '2h†']),
    'heat_treatment': ('Heat-treatment', ['None', 'Homogenized']),
    'forging_temp': ('Forging temperature', ['250°C', '300°C', '350°C']),
    'magnification': ('Magnification', ['100x', '500x', '1000x', '1500x', '2000x', '3000x']),
}
SOAKING_FOOTNOTE = '* 1.5h at 350°C + 1.5h at 250°C      † 2h at 350°C + 1h at 250°C'

# Optional scale bars: {magnification code: (label, bar length in pixels of the
# 512-px image)}. The dataset images carry no scale information, so none are
# drawn unless you fill this in from your microscope calibration, e.g.
# SCALE_BARS = {2: ('10 µm', 98), 4: ('2 µm', 80)}
SCALE_BARS = {}

_series_paths = {}
for _ds in (train_dataset, test_dataset):
    for _p, _s in _ds.samples:
        _series_paths.setdefault(_s, []).append(_p)


def series_conditions(series):
    table = train_labels if series in train_labels.index else test_labels
    row = table.loc[series]
    return {k: int(row[k]) for k in CONDITION_COLUMNS}


def condition_lines(cond):
    """Two columns of 'Name: value' strings, laid out like the paper's captions."""
    columns = [['shape', 'soaking', 'forging_temp'], ['cooling_rate', 'heat_treatment', 'magnification']]
    return [[f'{COND_TEXT[k][0]}: {COND_TEXT[k][1][cond[k]]}' for k in col] for col in columns]


def real_crop(series, size=IMAGE_SIZE):
    """Centre 512x512 crop of the first real image of a series, in [0, 1] (None
    if the labels list the series but no images were uploaded for it)."""
    paths = _series_paths.get(series)
    if not paths:
        return None
    arr = np.asarray(Image.open(paths[0]).convert('L'), dtype=np.float32) / 255
    h, w = arr.shape
    if h < size or w < size:
        arr = np.pad(arr, ((0, max(0, size - h)), (0, max(0, size - w))), mode='reflect')
        h, w = arr.shape
    top, left = (h - size) // 2, (w - size) // 2
    return arr[top:top + size, left:left + size]


# --- GPU memory ---------------------------------------------------------------
# At 512x512 this U-Net is ~377M parameters = ~1.4 GiB per copy in fp32 (82% of
# that is the per-block condition embeddings, whose `nn.Linear(100, imsize^2)`
# is 185M parameters in `up1` alone). After training, the GPU holds `model`,
# `ema_model` and Adam's two state tensors -- ~5.6 GiB -- and the figures below
# only ever run `ema_model`. Freeing the rest leaves room for sampling and for
# the InceptionV3 network the FID curve needs.
RELEASE_TRAINING_MEMORY = True


def gpu_allocated_gib():
    return torch.cuda.memory_allocated() / 2 ** 30 if torch.cuda.is_available() else 0.0


def release_training_memory():
    """Free the training-only tensors: Adam's state and the training model's weights.

    They are dropped, not moved: copying ~4.2 GiB to host RAM is what crashes a
    12.7 GB Colab runtime. `model` stays as a shape-only ("meta") module, so the
    Section 6 training loop cannot run in this session -- to train further,
    re-run Section 5 with `RESUME_FROM` set to your last checkpoint, which
    rebuilds them."""
    if not torch.cuda.is_available():
        return
    before = gpu_allocated_gib()
    optimizer.state.clear()            # Adam exp_avg / exp_avg_sq, ~2.8 GiB
    model.to_empty(device='meta')      # weights, ~1.4 GiB; no host copy is made
    gc.collect()
    torch.cuda.empty_cache()
    print(f'Released training-only GPU memory: {before:.2f} -> {gpu_allocated_gib():.2f} GiB allocated '
          f'(ema_model stays on the GPU).')


if RELEASE_TRAINING_MEMORY:
    release_training_memory()


# --- Generation (with an on-disk cache: sampling is the slow part) ------------
def _load_checkpoint(path):
    """Read a checkpoint onto the CPU.

    `map_location=device` would copy the whole file -- model + EMA model +
    Adam state, about 4x what is needed -- straight onto the GPU, which OOMs a
    15-16GB card once training has filled it. Reading to CPU and letting
    `load_state_dict` copy tensor by tensor keeps the GPU cost at zero, and
    `mmap=True` avoids reading the parts we never touch."""
    try:
        return torch.load(path, map_location='cpu', mmap=True)
    except (TypeError, RuntimeError, ValueError):
        return torch.load(path, map_location='cpu')


_ema_backup = None


@contextlib.contextmanager
def ema_weights_from(path):
    """Run `ema_model` with the EMA weights of `path`, restoring it afterwards.

    Swapping weights in place avoids keeping a second model on the GPU (~1.4
    GiB). The trained weights are backed up on the GPU on first use so the
    figures after this one still see the model the training loop produced."""
    global _ema_backup
    if _ema_backup is None:
        # Kept on the GPU (there is room once the training tensors are freed): a host
        # copy would add ~1.4 GiB of RAM for no benefit.
        _ema_backup = {k: v.detach().clone() for k, v in ema_model.state_dict().items()}
    ckpt = _load_checkpoint(path)
    iteration_ = int(ckpt.get('iteration', 0))
    ema_model.load_state_dict(ckpt['ema_model_state'])
    del ckpt
    try:
        yield ema_model, iteration_
    finally:
        ema_model.load_state_dict(_ema_backup)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def current_iteration():
    return int(globals().get('iteration', start_iteration - 1))


@contextlib.contextmanager
def figure_model():
    """(model, cache tag) for the weights the figures should be drawn from."""
    if FIGURE_MODEL_CHECKPOINT is None:
        yield ema_model, f'iter{current_iteration():06d}'
    else:
        with ema_weights_from(FIGURE_MODEL_CHECKPOINT) as (source_model, iteration_):
            yield source_model, f'iter{iteration_:06d}'


def model_fingerprint(m):
    """[FIX] short hash of the weights, used in the image-cache key so that a retrained (or different)
    model never reuses another model's cached images."""
    with torch.no_grad():
        s = sum(float(torch.sum(p.detach(), dtype=torch.float64)) for p in m.parameters())
    return f'{s:.6e}'


def generate_images(source_model, conds, seed=None, cache_tag=None):
    """conds: list of 7-key condition dicts. Returns float [N, 1, H, W] in [0, 1].

    Each chunk of GEN_BATCH_SIZE images is cached as uint8 under
    PAPER_CACHE_DIR (keyed by cache_tag + the chunk's conditions + seed), so a
    disconnected session or a re-run doesn't repeat ~1000 denoising steps."""
    outs = []
    fp = model_fingerprint(source_model) if cache_tag is not None else None
    for k, i in enumerate(range(0, len(conds), GEN_BATCH_SIZE)):
        chunk = conds[i:i + GEN_BATCH_SIZE]
        chunk_seed = None if seed is None else seed + k
        cache_file = None
        if cache_tag is not None:
            key = hashlib.md5(json.dumps([chunk, chunk_seed, fp]).encode()).hexdigest()[:10]
            cache_file = PAPER_CACHE_DIR / f'{cache_tag}_{key}.npy'
            if cache_file.exists():
                outs.append(torch.from_numpy(np.load(cache_file).astype(np.float32) / 255))
                continue
        if chunk_seed is not None:
            torch.manual_seed(chunk_seed)
        cond = torch.tensor([[c[name] for name in CONDITION_COLUMNS] for c in chunk],
                            dtype=torch.long, device=device)
        shp, loc, cr, sk, ht, ft, mag = cond.unbind(dim=1)
        x = diffusion.sample(source_model, len(chunk), shp, loc, cr, sk, ht, ft, mag)
        x = ((x.clamp(-1, 1) + 1) / 2).cpu()
        if cache_file is not None:
            np.save(cache_file, (x.numpy() * 255).round().astype(np.uint8))
            x = torch.from_numpy(np.load(cache_file).astype(np.float32) / 255)  # same 8-bit values as a cache hit
        outs.append(x)
    return torch.cat(outs)


def synthesize_series(series_list, source_model, tag, seed=SEED):
    """One synthesized image per series, as 2-D float arrays in [0, 1]."""
    imgs = generate_images(source_model, [series_conditions(s) for s in series_list],
                           seed=seed, cache_tag=f'{tag}_series')
    return {s: im[0].numpy() for s, im in zip(series_list, imgs)}


# --- Approximate phase highlighting ------------------------------------------
# The paper's coloured Mg17Al12 morphology labels (A eutectic, B intergranular,
# C globular, D continuous) were annotated by hand. This is an automatic,
# rule-based stand-in (brightness threshold + particle size/shape) applied
# identically to real and synthesized images. It is only approximate -- it
# misses mid-grey particles and mislabels some -- so treat it as a visual aid.
HIGHLIGHT_PHASES = True
PHASES = {
    'A': ('Eutectic Mg$_{17}$Al$_{12}$', (0.91, 0.60, 0.60)),
    'B': ('Intergranular Mg$_{17}$Al$_{12}$', (0.18, 0.62, 0.28)),
    'C': ('Globular Mg$_{17}$Al$_{12}$', (0.96, 0.77, 0.10)),
    'D': ('Continuous Mg$_{17}$Al$_{12}$', (0.69, 0.36, 0.75)),
}


def segment_phases(img01):
    """Returns (label map with 0 = matrix and 1..4 = A..D, {letter: (row, col)
    of the largest particle of that class})."""
    g = ndi.gaussian_filter(img01, 1.0)
    matrix = np.median(g)
    spread = np.median(np.abs(g - matrix)) * 1.4826 + 1e-6
    bright = g > matrix + 4.0 * spread  # Mg17Al12 is the brightest phase
    bright = ndi.binary_fill_holes(ndi.binary_opening(bright))
    lab = measure.label(bright, connectivity=2)
    props = [p for p in measure.regionprops(lab) if p.area >= 15]
    out = np.zeros(img01.shape, np.uint8)
    small = np.array([p.centroid for p in props if p.area < 150])
    tree = cKDTree(small) if len(small) > 1 else None
    largest = {}
    for p in props:
        elong = p.axis_major_length / max(p.axis_minor_length, 1.0)
        if p.area >= 500:
            k = 1
        elif tree is not None and elong > 1.8 and len(tree.query_ball_point(p.centroid, 30)) >= 12:
            k = 4  # dense field of fine elongated particles
        elif elong > 3.5:
            k = 2  # thin and elongated (grain-boundary films)
        elif p.solidity > 0.8 and elong < 2.2:
            k = 3  # compact, roundish
        else:
            continue
        out[lab == p.label] = k
        if k not in largest or p.area > largest[k][0]:
            largest[k] = (p.area, p.centroid)
    return out, {'ABCD'[k - 1]: c for k, (_, c) in largest.items()}


def phase_overlay(img01, seg, alpha=0.6):
    rgb = np.stack([img01] * 3, axis=-1)
    for k, letter in enumerate('ABCD', 1):
        m = seg == k
        rgb[m] = (1 - alpha) * rgb[m] + alpha * np.array(PHASES[letter][1])
    return rgb


# --- Drawing ------------------------------------------------------------------
def draw_ibeam(ax, location):
    """I-beam pictogram as in the paper's figures; the red square marks where the
    metallography sample was cut (0 tall flange, 1 web, 2 short flange)."""
    tan = '#B9AC97'
    ax.add_patch(Rectangle((1.2, -0.45), 7.6, 0.9, color=tan, lw=0))   # web
    ax.add_patch(Rectangle((0.0, -2.6), 1.6, 5.2, color=tan, lw=0))    # tall flange
    ax.add_patch(Rectangle((8.4, -1.7), 1.6, 3.4, color=tan, lw=0))    # short flange
    cx = {0: 0.8, 1: 5.0, 2: 9.2}[location]
    ax.add_patch(Rectangle((cx - 0.55, -0.55), 1.1, 1.1, color='#D62828', lw=0))
    ax.set_xlim(-0.3, 10.3)
    ax.set_ylim(-3, 3)
    ax.set_aspect('equal')
    ax.axis('off')


def show_image(ax, arr, tag=None, highlight=False, magnification=None):
    """Draw one micrograph; `tag` ('Synthesized' / 'Real') goes in the corner."""
    if arr is None:
        ax.text(0.5, 0.5, 'no real image\nfor this series', ha='center', va='center',
                transform=ax.transAxes, color='#666')
        ax.set_facecolor('#eee')
    elif highlight:
        seg, marks = segment_phases(arr)
        ax.imshow(phase_overlay(arr, seg))
        for letter, (r, c) in marks.items():
            ax.text(c, r, letter, color='white', fontsize=11, fontweight='bold', ha='center', va='center',
                    path_effects=[pe.withStroke(linewidth=2, foreground='black')])
    else:
        ax.imshow(arr, cmap='gray', vmin=0, vmax=1)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    if tag:
        ax.text(0.02, 0.98, tag, transform=ax.transAxes, color='white', fontsize=9, fontweight='bold',
                va='top', bbox=dict(facecolor='#222', alpha=0.75, pad=2, lw=0))
    if arr is not None and magnification in SCALE_BARS:
        label, length = SCALE_BARS[magnification]
        h, w = arr.shape
        ax.add_patch(Rectangle((w - length - 14, h - 30), length, 7, color='white'))
        ax.text(w - 14 - length / 2, h - 36, label, color='white', fontsize=8, fontweight='bold', ha='center',
                path_effects=[pe.withStroke(linewidth=2, foreground='black')])


def phase_legend(fig, highlight):
    """Legend + footnotes strip for a figure (or subfigure)."""
    if highlight:
        handles = [Patch(facecolor=color, label=f'{letter}   {name}') for letter, (name, color) in PHASES.items()]
        fig.legend(handles=handles, loc='upper center', ncol=4, frameon=True, fontsize=9,
                   bbox_to_anchor=(0.5, 0.95))
    note = SOAKING_FOOTNOTE
    if highlight:
        note += '\nPhase colours: automatic brightness/shape heuristic (approximate; the paper\'s were annotated by hand).'
    fig.text(0.5, 0.02, note, ha='center', va='bottom', fontsize=8, color='#444')

<div dir="rtl">

### ۱۰-۱. شکل ۲: روند آموزش و FID

**(a)** FID بین تصاویر مصنوعی و واقعی در هر checkpoint (زرد) در برابر خط مبنای «واقعی در برابر واقعی» (آبی، کمترین FID قابل‌دسترس) و در صورت تنظیم `STEEL_IMAGE_DIR`، مرجع «فولاد در برابر واقعی» (خاکستری). **(b تا h)** از یک نویز اولیهٔ ثابت، وزن‌های EMA چند checkpoint پشت‌سرهم تصویر را می‌سازند تا بتوانی یک تصویر را در حال شفاف‌شدن در طول آموزش ببینی. FID با `pytorch-fid` (InceptionV3) حساب می‌شود و وزن‌های آن در اولین استفاده دانلود می‌شوند، پس اینترنت لازم است؛ بدون آن فقط تصاویر پیشرفت کشیده می‌شوند.

checkpointها در تکرار ۳۰ و هر ۶۰۰ تکرار ذخیره می‌شوند و همان پنل‌های b تا h مقاله‌اند (۳۰، ۶۰۰، ۱۲۰۰، …، ۳۶۰۰).

</div>

In [ ]:
# ======================================================================
# PAPER MAP | Fig. 2 -- dependency for the FID curve (pytorch-fid, InceptionV3 features)

# ======================================================================

# --- Dependency for the FID curve (Fig. 2): pytorch-fid ----------------------
# The paper computes FID with Seitzer's pytorch-fid (an InceptionV3 feature
# extractor). It is NOT preinstalled on Kaggle/Colab, and its InceptionV3
# weights are downloaded on first use -- so this needs internet access
# (Kaggle: Settings -> Internet -> On). If it can't be installed, Fig. 2's
# progress images are still drawn; only the FID curve is skipped.
try:
    from pytorch_fid.inception import InceptionV3
except ImportError:
    import subprocess
    import sys
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pytorch-fid'])
        from pytorch_fid.inception import InceptionV3
    except Exception as e:
        InceptionV3 = None
        print(f'pytorch-fid unavailable ({type(e).__name__}); the FID curve will be skipped.')
print('FID available:', InceptionV3 is not None)

In [ ]:
# ======================================================================
# PAPER MAP | Section 2.2.4 and Fig. 2 -- FID vs. training iteration + one image sharpening over training
# FID = ||mu_r-mu_g||^2 + Tr(S_r + S_g - 2 (S_r S_g)^(1/2)) on InceptionV3 features (pytorch-fid).
# Baseline = FID between two disjoint subsets of REAL images (paper: lowest attainable value).
# ======================================================================

# --- 9.1  Fig. 2: FID vs. training iteration + generation progress -----------
# What is computed: for each saved checkpoint, generate FID_N_SYNTH images
# (conditions drawn from the training set), and compare their InceptionV3
# features with real training crops. Baseline = two disjoint subsets of REAL
# images (the lowest FID the data itself allows). All sets use the same size,
# because FID is biased upward for small samples.
#
# Runtime warning: every synthesized image is ~1000 U-Net passes, so cost is
# roughly (checkpoints x FID_N_SYNTH + checkpoints) images. Results are cached
# in PAPER_CACHE_DIR, so an interrupted run resumes where it stopped.
FID_N_SYNTH = 4 if SMOKE_TEST else 16            # per checkpoint. The paper's absolute FID values need far more
                            # images; with few images use the curve's SHAPE, not its numbers.
FID_BATCH = 4               # images per InceptionV3 pass (lower it if you still hit OOM)
FID_REPEATS = 5             # random real subsets averaged for each FID value / the baseline
FIG2_MAX_PANELS = 7         # progress images shown (paper: iterations 30, 600, ..., 3600)
FIG2_PROGRESS_SERIES = 'CM01-1000'  # condition shown in the progress strip (any series name)
STEEL_IMAGE_DIR = None      # optional: folder of steel SEM images (paper ref. [54]) for the grey
                            # "Steel vs. Real" reference line; None skips it.

YELLOW, BLUE, GREY = '#F2B705', '#1f9bf0', '#9a9a9a'


def list_checkpoints():
    found = []
    for p in sorted(CHECKPOINT_DIR.glob('ckpt_iter*.pth.tar')):
        m = re.search(r'iter(\d+)', p.name)
        if m:
            found.append((int(m.group(1)), p))
    return sorted(found)


def pick_checkpoints(found, max_n):
    """Keep the first and last checkpoint and spread the rest evenly."""
    if len(found) <= max_n:
        return found
    idx = np.unique(np.round(np.linspace(0, len(found) - 1, max_n)).astype(int))
    return [found[i] for i in idx]


def frechet_distance(f1, f2):
    """FID between two feature sets (rows = images). Uses the identity
    tr sqrt(S1 S2) = sum of singular values of (A B^T) / sqrt((n-1)(m-1)) with
    A, B the centred features, which is exact and avoids a 2048x2048 sqrtm."""
    a, b = f1 - f1.mean(0), f2 - f2.mean(0)
    n, m = len(f1), len(f2)
    tr_sqrt = np.linalg.svd(a @ b.T, compute_uv=False).sum() / np.sqrt((n - 1) * (m - 1))
    d = f1.mean(0) - f2.mean(0)
    return float(d @ d + (a ** 2).sum() / (n - 1) + (b ** 2).sum() / (m - 1) - 2 * tr_sqrt)


_fid_net = None


@torch.no_grad()
def inception_features(images01, batch=FID_BATCH):
    """images01: float [N, 1, H, W] in [0, 1] -> [N, 2048] InceptionV3 pool features."""
    global _fid_net
    if _fid_net is None:
        _fid_net = InceptionV3([InceptionV3.BLOCK_INDEX_BY_DIM[2048]]).to(device).eval()
    feats = []
    for i in range(0, len(images01), batch):
        x = images01[i:i + batch].to(device).float().clamp(0, 1).repeat(1, 3, 1, 1)
        feats.append(_fid_net(x)[0].flatten(1).cpu().numpy().astype(np.float64))
    return np.concatenate(feats)


def free_fid_model():
    """Drop InceptionV3 once the curve is computed -- the figures after this one
    need the GPU for sampling."""
    global _fid_net
    _fid_net = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def mean_std_fid(feats_a, feats_b, n, repeats, rng, disjoint=False):
    """Mean/std of FID between random size-n subsets of two feature pools. With
    disjoint=True (same pool passed twice) the two subsets do not overlap."""
    vals = []
    for _ in range(repeats):
        if disjoint:
            perm = rng.permutation(len(feats_a))
            a, b = feats_a[perm[:n]], feats_a[perm[n:2 * n]]
        else:
            a = feats_a[rng.choice(len(feats_a), n, replace=False)]
            b = feats_b
        vals.append(frechet_distance(a, b))
    return float(np.mean(vals)), float(np.std(vals))


def load_real_pool(n, rng):
    """n random real training crops as float [n, 1, H, W] in [0, 1]."""
    idx = rng.choice(len(train_dataset), size=n, replace=False)
    return torch.stack([(train_dataset[int(i)][0] + 1) / 2 for i in idx])


def load_steel_pool(folder, n, rng):
    files = [p for p in sorted(Path(folder).rglob('*')) if p.suffix.lower() in AZ80SeriesDataset.IMG_EXTS]
    if len(files) < n:
        raise RuntimeError(f'Need at least {n} steel images in {folder}, found {len(files)}.')
    crops = []
    for i in rng.choice(len(files), size=n, replace=False):
        arr = np.asarray(Image.open(files[int(i)]).convert('L'), dtype=np.float32) / 255
        h, w = arr.shape
        if h < IMAGE_SIZE or w < IMAGE_SIZE:
            arr = np.pad(arr, ((0, max(0, IMAGE_SIZE - h)), (0, max(0, IMAGE_SIZE - w))), mode='reflect')
            h, w = arr.shape
        r, c = rng.randint(0, h - IMAGE_SIZE + 1), rng.randint(0, w - IMAGE_SIZE + 1)
        crops.append(torch.from_numpy(arr[r:r + IMAGE_SIZE, c:c + IMAGE_SIZE])[None])
    return torch.stack(crops)


def compute_fid_curve(checkpoints):
    """Returns dict with per-checkpoint FID (mean/std) and the reference lines."""
    try:
        return _compute_fid_curve(checkpoints)
    finally:
        free_fid_model()


def _compute_fid_curve(checkpoints):
    rng = np.random.RandomState(SEED)
    n = min(FID_N_SYNTH, len(train_dataset) // 2)
    real_pool = load_real_pool(2 * n, rng)
    real_feats = inception_features(real_pool)
    res = {'n': n, 'iters': [], 'mean': [], 'std': []}
    res['baseline'] = mean_std_fid(real_feats, real_feats, n, FID_REPEATS, rng, disjoint=True)
    res['steel'] = None
    if STEEL_IMAGE_DIR is not None:
        steel_feats = inception_features(load_steel_pool(STEEL_IMAGE_DIR, n, rng))
        res['steel'] = mean_std_fid(real_feats, steel_feats, n, FID_REPEATS, rng)
    synth_series = [train_dataset.samples[int(i)][1] for i in rng.choice(len(train_dataset), n, replace=False)]
    conds = [series_conditions(s) for s in synth_series]
    for it, path in checkpoints:
        with ema_weights_from(path) as (source_model, _):
            imgs = generate_images(source_model, conds, seed=SEED, cache_tag=f'fid_iter{it:06d}')
        m, s = mean_std_fid(real_feats, inception_features(imgs), n, FID_REPEATS, rng)
        res['iters'].append(it)
        res['mean'].append(m)
        res['std'].append(s)
        print(f'iter {it:>6}: FID = {m:.1f} +/- {s:.1f}  ({n} synthesized vs {n} real images, '
              f'{gpu_allocated_gib():.2f} GiB on GPU)')
    return res


def progress_images(checkpoints):
    cond = series_conditions(FIG2_PROGRESS_SERIES)
    out = []
    for it, path in checkpoints:
        with ema_weights_from(path) as (source_model, _):
            # Same seed for every checkpoint -> the SAME starting noise, so the
            # panels show one image sharpening as training proceeds (like the paper).
            img = generate_images(source_model, [cond], seed=SEED, cache_tag=f'progress_iter{it:06d}')
        out.append(img[0, 0].numpy())
    return out


def plot_fig2(res, images, iters, save_path):
    fig = plt.figure(figsize=(13, 7.9))
    gs = fig.add_gridspec(3, 5, wspace=0.07, hspace=0.07)
    # Paper layout: b..f along the top, g and h stacked under f, the FID plot
    # on the lower left (a).
    slots = [gs[0, 0], gs[0, 1], gs[0, 2], gs[0, 3], gs[0, 4], gs[1, 4], gs[2, 4]]
    axes = []
    for k, (img, it) in enumerate(zip(images, iters)):
        ax = fig.add_subplot(slots[k])
        ax.imshow(img, cmap='gray', vmin=0, vmax=1)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.text(0.03, 0.97, chr(ord('b') + k), transform=ax.transAxes, color='white', fontsize=12,
                fontweight='bold', va='top', bbox=dict(facecolor='black', alpha=0.65, pad=2, lw=0))
        ax.text(0.97, 0.03, f'iter {it}', transform=ax.transAxes, color='white', fontsize=8,
                ha='right', va='bottom', bbox=dict(facecolor='black', alpha=0.65, pad=1.5, lw=0))
        axes.append(ax)

    axp = fig.add_subplot(gs[1:3, 0:4])
    axp.spines[['top', 'right']].set_visible(False)
    axp.set_xlabel('Iteration', fontsize=12)
    axp.set_ylabel('FID Score', fontsize=12)
    axp.text(-0.09, 1.02, 'a', transform=axp.transAxes, fontsize=14, fontweight='bold')
    if res is None:
        axp.text(0.5, 0.5, 'FID curve skipped (pytorch-fid not installed)', ha='center', va='center',
                 transform=axp.transAxes, color='#666')
    else:
        xs, ys, sd = np.array(res['iters']), np.array(res['mean']), np.array(res['std'])
        axp.fill_between(xs, ys - sd, ys + sd, color=YELLOW, alpha=0.25, lw=0)
        axp.plot(xs, ys, '-', color=YELLOW, lw=2.5, label='Synthesized vs. Real')
        axp.plot(xs, ys, 'o', color='black', ms=5)
        for k, (x, y) in enumerate(zip(xs, ys)):
            axp.annotate(chr(ord('b') + k), (x, y), xytext=(0, 9), textcoords='offset points',
                         ha='center', fontsize=10, fontweight='bold')
        xmax = max(xs.max() * 1.1, 1)
        for key, color, label, text in [
            ('baseline', BLUE, 'Real vs. Real (baseline)', 'minimum attainable FID score for Real vs. Real'),
            ('steel', GREY, 'Steel vs. Real', 'minimum attainable FID score for steel vs. Real'),
        ]:
            if res[key] is None:
                continue
            m, s = res[key]
            axp.axhline(m, color=color, ls=':', lw=2, label=label)
            axp.axhspan(m - s, m + s, color=color, alpha=0.15, lw=0)
            axp.text(xmax / 2, m - s, text, color=color, ha='center', va='top', fontsize=9, fontweight='bold')
        axp.set_xlim(-xmax * 0.02, xmax)
        top = max((ys + sd).max(), *(res[k][0] + res[k][1] for k in ('baseline', 'steel') if res[k]))
        axp.set_ylim(0, top * 1.3)  # headroom so the legend clears the curve
        axp.legend(loc='upper right', fontsize=9, frameon=True)
        axp.text(0.5, -0.17, f'FID from {res["n"]} synthesized vs {res["n"]} real images per point '
                 '(few images: read the trend, not the absolute values)', transform=axp.transAxes,
                 ha='center', fontsize=8, color='#555')

    # Arrows between the progress panels: b -> c -> d -> e -> f, then f -> g -> h.
    for ax in axes:
        ax.apply_aspect()
    def edge(ax, side):
        b = ax.get_position()
        return {'r': (b.x1, (b.y0 + b.y1) / 2), 'l': (b.x0, (b.y0 + b.y1) / 2),
                'b': ((b.x0 + b.x1) / 2, b.y0), 't': ((b.x0 + b.x1) / 2, b.y1)}[side]
    for k in range(len(axes) - 1):
        a, b = axes[k], axes[k + 1]
        p, q = (edge(a, 'r'), edge(b, 'l')) if k < 4 else (edge(a, 'b'), edge(b, 't'))
        fig.add_artist(FancyArrowPatch(
            p, q, transform=fig.transFigure, arrowstyle='-|>', mutation_scale=12, color='black', lw=1))
    fig.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()
    plt.close(fig)


ckpts = pick_checkpoints(list_checkpoints(), FIG2_MAX_PANELS)
if not ckpts:
    raise FileNotFoundError(f'No checkpoints found in {CHECKPOINT_DIR}. Train first (section 6) or set CHECKPOINT_DIR.')
print('Using checkpoints at iterations:', [it for it, _ in ckpts])

fid_res = compute_fid_curve(ckpts) if InceptionV3 is not None else None
prog = progress_images(ckpts)
plot_fig2(fid_res, prog, [it for it, _ in ckpts], PAPER_FIG_DIR / 'fig2_training_progress.png')

<div dir="rtl">

### ۱۰-۲. شکل ۳: تصویر مصنوعی و واقعی، شرایط دیده‌شده

هشت مسیر فرایند که در آموزش بوده‌اند: هر کدام تصویر مصنوعی کنار تصویر واقعی همان series، با هفت پارامتر فرایند زیر آن‌ها و یک نمادِ تیرچه که مربع قرمزش محل برش نمونه را نشان می‌دهد (بال بلند، جان یا بال کوتاه). شرط هر پنل a تا h از شکل ۳ مقاله برداشته شده است؛ با تغییر `FIG3_SERIES` می‌توانی series دیگری از `Training/` را ببینی.

</div>

In [ ]:
# ======================================================================
# PAPER MAP | Fig. 3 -- synthesized vs. real, eight SEEN conditions (panels a-h of the paper)

# ======================================================================

# --- 9.2  Fig. 3: synthesized vs. real, one panel per condition ---------------
# Panels a-h follow the paper's Fig. 3. Each is a series from the TRAINING set
# (seen conditions). Swap in any series names from `train_labels.index`.
FIG3_SERIES = [
    'CM01-1000',   # a  cylinder, 6.0 C/s, normal soak, no HT, 350 C, web,          1000x
    'PS03-2000',   # b  preform, 10.4 C/s, normal soak, homogenized, 300 C, short,  2000x
    'PL13-0500',   # c  preform, 10.4 C/s, 1.5h soak, no HT, 250 C, tall,            500x
    'CM10-1000',   # d  cylinder, 1.5 C/s, normal soak, no HT, 300 C, web,          1000x
    'CM16-3000',   # e  cylinder, 1.5 C/s, normal soak, no HT, 250 C, web,          3000x
    'PM16-1000',   # f  preform, 10.4 C/s, 2h soak, no HT, 250 C, web,              1000x
    'CM13-0500',   # g  cylinder, 6.0 C/s, normal soak, no HT, 250 C, web,           500x
    'PS11-2000',   # h  preform, 10.4 C/s, normal soak, no HT, 350 C, short,        2000x
]


def draw_pair_panel(sub, letter, series, synth, real, highlight):
    cond = series_conditions(series)
    gs = sub.add_gridspec(2, 2, height_ratios=[6, 1.8], wspace=0.03, hspace=0.08,
                          left=0.02, right=0.98, top=0.93, bottom=0.02)
    show_image(sub.add_subplot(gs[0, 0]), synth, 'Synthesized', highlight, cond['magnification'])
    show_image(sub.add_subplot(gs[0, 1]), real, 'Real', highlight, cond['magnification'])
    info = gs[1, :].subgridspec(1, 3, width_ratios=[0.7, 1.6, 1.6], wspace=0.02)
    draw_ibeam(sub.add_subplot(info[0]), cond['location'])
    for col, lines in enumerate(condition_lines(cond)):
        ax = sub.add_subplot(info[col + 1])
        ax.axis('off')
        ax.text(0, 0.5, '\n'.join(lines), va='center', ha='left', fontsize=8.5, linespacing=1.5)
    sub.text(0.0, 0.995, letter, fontsize=15, fontweight='bold', va='top')
    sub.text(0.5, 0.995, series, fontsize=8, color='#777', va='top', ha='center')


def make_fig3(series_list, highlight=HIGHLIGHT_PHASES):
    with figure_model() as (source_model, tag):
        synth = synthesize_series(series_list, source_model, tag)
    rows = (len(series_list) + 1) // 2
    fig = plt.figure(figsize=(12, 4.15 * rows + 1.0))
    body, foot = fig.subfigures(2, 1, height_ratios=[4.15 * rows, 1.0], hspace=0)
    panels = body.subfigures(rows, 2, wspace=0.03, hspace=0.03, squeeze=False)
    for k, series in enumerate(series_list):
        draw_pair_panel(panels[k // 2][k % 2], chr(ord('a') + k), series, synth[series],
                        real_crop(series), highlight)
    phase_legend(foot, highlight)
    fig.savefig(PAPER_FIG_DIR / 'fig3_synthesized_vs_real.png', dpi=200)
    plt.show()
    plt.close(fig)


make_fig3(FIG3_SERIES)

<div dir="rtl">

### ۱۰-۳. شکل ۴: پیش‌بینی برای شرایط دیده‌نشده

نکتهٔ اصلی مقاله: شرایطی که از آموزش کنار گذاشته شده‌اند (همهٔ series ها از `Testing/` می‌آیند). **(a)** خروج یک قطعهٔ کامل: همهٔ نمونه‌های یک تیرچه (جان و بال کوتاه با بزرگ‌نمایی 500x و 1500x). **(b)** خروج یک نمونهٔ متالوگرافی از قطعه‌ای دیگر (500x و 3000x). **(c)** خروج یک بزرگ‌نمایی از دو نمونهٔ دیگر (1500x و 1000x). تصاویر واقعی اینجا دادهٔ نگه‌داشته‌شده‌اند و مدل هرگز آن‌ها را ندیده است؛ پس شباهت تصویر مصنوعی به آن‌ها پیش‌بینی واقعی است.

</div>

In [ ]:
# ======================================================================
# PAPER MAP | Fig. 4 -- predictions for UNSEEN conditions (one-I-beam-out, one-sample-out, one-magnification-out)

# ======================================================================

# --- 9.3  Fig. 4: predictions for UNSEEN conditions ---------------------------
# The three leave-out scenarios from the paper -- all series below live in
# `Testing/`, so the model never saw them during training:
#   a  one-I-beam-out       every sample of one component (web + short flange)
#   b  one-sample-out       one metallography sample of another component
#   c  one-magnification-out  one magnification of two other samples
# Each block: a list of groups, each group = (series shown side by side).
FIG4_BLOCKS = [
    ('a', 'Unseen Component',           [['CM06-0500', 'CM06-1500'], ['CS06-0500', 'CS06-1500']]),
    ('b', 'Unseen Metallography Sample', [['PS13-0500', 'PS13-3000']]),
    ('c', 'Unseen Magnification',       [['PM03-1500'], ['CS04-1000']]),
]
FIG4_COLS = 4  # image columns per row; blocks with fewer are centred


def draw_block(sub, letter, title, groups, synth, highlight):
    n_cols = sum(len(g) for g in groups)
    gs = sub.add_gridspec(5, FIG4_COLS, height_ratios=[0.4, 1.5, 0.35, 5, 5], wspace=0.04, hspace=0.06,
                          left=0.01, right=0.99, top=0.99, bottom=0.01)
    sub.add_subplot(gs[0, :]).axis('off')
    sub.text(0.005, 0.995, f'{letter}', fontsize=15, fontweight='bold', va='top')
    sub.text(0.035, 0.99, title, fontsize=13, fontweight='bold', va='top', color='#1f3a93')
    col = (FIG4_COLS - n_cols) // 2
    for group in groups:
        width = len(group)
        cond = series_conditions(group[0])
        hdr = sub.add_subplot(gs[1, col:col + width])
        hdr.axis('off')
        draw_ibeam(hdr.inset_axes([0.0, 0.05, 0.2 if width > 1 else 0.24, 0.9]), cond['location'])
        left, right = condition_lines(cond)
        # Magnification is shown in the boxes under the header, one per column.
        lines = [left, [ln for ln in right if not ln.startswith('Magnification')]]
        if width > 1:
            for c, txt in enumerate(lines):
                hdr.text(0.26 + 0.38 * c, 0.5, '\n'.join(txt), va='center', ha='left', fontsize=8, linespacing=1.5)
        else:
            hdr.text(0.28, 0.5, '\n'.join(lines[0] + lines[1]), va='center', ha='left', fontsize=7.5,
                     linespacing=1.35)
        for j, series in enumerate(group):
            c_j = series_conditions(series)
            box = sub.add_subplot(gs[2, col + j])
            box.axis('off')
            box.text(0.5, 0.5, COND_TEXT['magnification'][1][c_j['magnification']], ha='center', va='center',
                     fontsize=10, fontweight='bold', color='#1f3a93',
                     bbox=dict(boxstyle='square,pad=0.3', fc='white', ec='#1f3a93'))
            show_image(sub.add_subplot(gs[3, col + j]), synth[series], 'Synthesized', highlight,
                       c_j['magnification'])
            show_image(sub.add_subplot(gs[4, col + j]), real_crop(series), 'Real', highlight,
                       c_j['magnification'])
        col += width


def make_fig4(blocks, highlight=HIGHLIGHT_PHASES):
    all_series = [s for _, _, groups in blocks for g in groups for s in g]
    with figure_model() as (source_model, tag):
        synth = synthesize_series(all_series, source_model, tag)
    block_h = 7.6
    fig = plt.figure(figsize=(12, block_h * len(blocks) + 1.0))
    body, foot = fig.subfigures(2, 1, height_ratios=[block_h * len(blocks), 1.0], hspace=0)
    subs = body.subfigures(len(blocks), 1, hspace=0.02, squeeze=False)
    for k, (letter, title, groups) in enumerate(blocks):
        draw_block(subs[k][0], letter, title, groups, synth, highlight)
    phase_legend(foot, highlight)
    fig.savefig(PAPER_FIG_DIR / 'fig4_unseen_conditions.png', dpi=200)
    plt.show()
    plt.close(fig)


make_fig4(FIG4_BLOCKS)